## 1. 安裝套件


In [ ]:
'''
Install ComfyUI and the custom nodes used by the LTX-2.5 workflow.
安裝 LTX-2.5 Workflow 會用到的 ComfyUI 與自訂節點。
'''
import os
import subprocess
import sys
from pathlib import Path
import torch

ROOT = Path('/kaggle/working')
COMFY_DIR = ROOT / 'ComfyUI'

'''
Select the latest verified stable ComfyUI release.
指定已確認的最新 ComfyUI 穩定版。
'''
COMFY_REF = 'v0.45.0'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

repos = [
    ('https://github.com/Comfy-Org/ComfyUI.git', COMFY_DIR, COMFY_REF),
    ('https://github.com/city96/ComfyUI-GGUF.git', COMFY_DIR / 'custom_nodes/ComfyUI-GGUF', None),
    ('https://github.com/Lightricks/ComfyUI-LTXVideo.git', COMFY_DIR / 'custom_nodes/ComfyUI-LTXVideo', None),
    ('https://github.com/kijai/ComfyUI-KJNodes.git', COMFY_DIR / 'custom_nodes/ComfyUI-KJNodes', None),
    ('https://github.com/yolain/ComfyUI-Easy-Use.git', COMFY_DIR / 'custom_nodes/ComfyUI-Easy-Use', None),
    ('https://github.com/SethRobinson/comfyui-workflow-to-api-converter-endpoint.git', COMFY_DIR / 'custom_nodes/workflow-to-api-converter', None),
]

'''
Update extension code without replacing local edits.
更新擴充節點程式，但不覆寫本機修改。
'''
UPDATE_CUSTOM_NODES = True

'''
Remember exact revisions to detect running-server incompatibilities.
記錄實際 commit，避免服務仍使用舊版節點。
'''
original_revisions = {}

'''
Install missing repositories and safely update existing ones.
安裝缺少的倉庫，並安全更新已存在的倉庫。
'''
for url, directory, ref in repos:

    '''
    Require an existing folder to be a proper Git repository.
    已存在的目錄必須是有效的 Git 倉庫。
    '''
    if directory.exists() and not (directory / '.git').exists():

        '''
        Do not overwrite unknown user files.
        不覆寫來源不明的使用者檔案。
        '''
        raise RuntimeError(f'已有非 Git 資料夾：{directory}')

    '''
    Remember the old commit if this package was already installed.
    記錄原先已安裝套件的版本。
    '''
    if (directory / '.git').exists():

        '''
        Read the current checkout's exact commit.
        讀取目前 checkout 的確切 commit。
        '''
        original_revisions[directory] = subprocess.check_output(['git', '-C', str(directory), 'rev-parse', 'HEAD'], text=True).strip()
    else:

        '''
        Clone only missing source packages.
        只 Clone 尚未存在的程式庫。
        '''
        command = ['git', '-c', 'advice.detachedHead=false', 'clone', '--depth', '1', '-q']

        '''
        Select the stable release for ComfyUI itself.
        ComfyUI 核心固定於穩定發行版。
        '''
        if ref:

            '''
            Append the release tag for the initial checkout.
            在首次 Clone 時指定發行版標籤。
            '''
            command += ['--branch', ref]

        '''
        Install the repository in its original location.
        將程式庫安裝於原有位置。
        '''
        subprocess.run(command + [url, str(directory)], check=True)

    '''
    Update extensions without force-resetting their local files.
    更新外掛時不強制覆寫本機檔案。
    '''
    if directory != COMFY_DIR and UPDATE_CUSTOM_NODES:

        '''
        Update only clean checkouts to avoid merge conflicts.
        僅更新沒有本機修改的倉庫，避免合併衝突。
        '''
        if not subprocess.check_output(['git', '-C', str(directory), 'status', '--porcelain'], text=True).strip():

            '''
            Reject non-fast-forward Git merges.
            拒絕非 fast-forward 的 Git 合併。
            '''
            subprocess.run(['git', '-C', str(directory), 'pull', '--ff-only', '-q'], check=True)
        else:

            '''
            Preserve an extension containing user modifications.
            保留使用者已修改的外掛。
            '''
            print(f'略過已有本機修改的外掛：{directory.name}')

'''
Retrieve the requested release even for an existing checkout.
即使倉庫已存在，仍取得指定的正式發行標籤。
'''
subprocess.run(['git', '-C', str(COMFY_DIR), 'fetch', '--depth', '1', 'origin', 'tag', COMFY_REF], check=True)

'''
Use the verified tag rather than a moving default branch.
使用經確認的標籤，而非隨時變動的預設分支。
'''
subprocess.run(['git', '-C', str(COMFY_DIR), 'checkout', '--quiet', '--detach', COMFY_REF], check=True)

'''
Indicate whether an existing model server must be restarted.
標記已啟動的模型服務是否需要重新啟動。
'''
LTX_CODE_UPDATED = any(
    directory in original_revisions and original_revisions[directory] != subprocess.check_output(
        ['git', '-C', str(directory), 'rev-parse', 'HEAD'], text=True,
    ).strip()
    for _, directory, _ in repos
)

'''
Record each exact source revision for troubleshooting.
記錄每個元件的原始碼版本以供除錯。
'''
for _, directory, _ in repos:

    '''
    Show the active release or extension commit.
    顯示目前使用的核心或外掛 commit。
    '''
    print(directory.name, subprocess.check_output(['git', '-C', str(directory), 'rev-parse', '--short', 'HEAD'], text=True).strip())

'''
Prevent requirements from replacing Kaggle's working PyTorch or switching to Transformers 5.
避免相依套件替換 Kaggle PyTorch 或切換至 Transformers 5。
'''
constraint = ROOT / 'ltx25_torch_constraint.txt'

'''
Save the interpreter-specific compatibility limits.
儲存目前 Python 環境適用的相容性限制。
'''
constraint.write_text(f'torch=={torch.__version__}\ntransformers>=4.50.3,<5.0\n', encoding='utf-8')

for _, directory, _ in repos:
    requirement = directory / 'requirements.txt'
    if requirement.is_file():
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-c', str(constraint), '-r', str(requirement)], check=True)

'''
Install download and interface packages under the same dependency constraints.
安裝下載與介面套件時沿用相同的相容性限制。
'''
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-c', str(constraint), 'huggingface_hub', 'hf_xet', 'requests', 'Pillow', 'ipywidgets'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', 'kornia==0.8.2'], check=True)

legacy = COMFY_DIR / 'custom_nodes/ComfyUI-MultiGPU'
if legacy.is_dir():
    legacy.rename(ROOT / 'ComfyUI-MultiGPU.disabled')

print('ComfyUI installed:', COMFY_DIR)
print('Torch:', torch.__version__)


## 2. 設定環境


In [ ]:
'''
Import the packages required by the notebook.
匯入 Notebook 所需套件。
'''
import importlib.util
import os
import shutil
import subprocess
import time
from pathlib import Path
import ipywidgets as widgets
import requests
import torch
from IPython.display import FileLink, Video, display
from PIL import Image

'''
Define the Kaggle workspace and ComfyUI paths.
設定 Kaggle 與 ComfyUI 的工作路徑。
'''
ROOT = Path('/kaggle/working')
COMFY_DIR = ROOT / 'ComfyUI'
FLOW_REGISTRY_PATH = ROOT / 'ltx25_local_flows.json'
COMFY_LOG = ROOT / 'comfyui_ltx25.log'
COMFY_PORT = 8188

'''
Store large model weights outside the 20 GB output partition.
將大型模型權重放在輸出磁區以外的暫存空間。
'''
SCRATCH_DIR = Path('/tmp/ltx25')
MODEL_STORE = SCRATCH_DIR / 'ltx25_models'

'''
Create the temporary model directories.
建立模型暫存資料夾。
'''
MODEL_STORE.mkdir(parents=True, exist_ok=True)

'''
Direct Hugging Face downloads and caches to scratch space.
將 Hugging Face 下載與快取指定至暫存空間。
'''
os.environ['HF_HOME'] = str(SCRATCH_DIR / 'huggingface')
os.environ['HF_XET_CACHE'] = str(SCRATCH_DIR / 'huggingface' / 'xet')
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

'''
Require Kaggle's two-GPU runtime before loading the workflow.
載入 Workflow 前，確認 Kaggle 已啟用雙 GPU。
'''
assert torch.cuda.device_count() == 2, '請在 Kaggle Settings 選擇 GPU T4 ×2。'

'''
Display the two detected GPU names.
顯示兩張 GPU 的名稱。
'''
for index in range(2):

    '''
    Show the active device name.
    顯示目前裝置名稱。
    '''
    print(f'GPU {index}: {torch.cuda.get_device_name(index)}')


## 3. 模型與 Workflow


In [ ]:
'''
Fetch only the model files used by this Q4_K_M workflow.
只下載 Q4_K_M 雙階段 Workflow 真正需要的模型。
'''
import json
import shutil
import wave
from huggingface_hub import hf_hub_download
from PIL import Image

'''
Keep the registry compatible with the original conversion engine.
保持模型登錄格式與原本的 Workflow 轉換程式相容。
'''
FLOW_REGISTRY = {
    "schema_version": 25,
    "models": {
        "transformers": {"q4km": {"filename": "LTX-2.5-Distilled-Q4_K_M.gguf"}},
        "text_encoder": {"basename": "gemma4-12b-ltx25-w4a8-v2.safetensors"},
        "video_vae": {"filename": "vae/ltx-2.5-video-vae-conv-bf16.safetensors"},
        "audio_vae": {"filename": "vae/ltx-2.5-audio-vae-bf16.safetensors"},
        "spatial_upscaler": {"filename": "latent_upscale_models/ltx-2.5-latent-spatial-upscaler-x2-bf16-1.0.safetensors"},
    },
    "workflows": {"t2v_i2v": {"local_name": "LTX25_T2V_I2V.json"}},
}

'''
Store the downloaded model weights in Kaggle scratch space.
將模型權重下載到 Kaggle 暫存空間，減少 working 配額占用。
'''
model_files = [
    ("realrebelai/LTX-2.5_GGUFs", "LTX-2.5-Distilled-Q4_K_M.gguf", "diffusion_models"),
    ("realrebelai/Rebels_w4a8s", "LTX/ENCODERS/gemma4-12b-ltx25-w4a8-v2.safetensors", "text_encoders"),
    ("comfyicu/LTX-2.5", "vae/ltx-2.5-video-vae-conv-bf16.safetensors", "vae"),
    ("comfyicu/LTX-2.5", "vae/ltx-2.5-audio-vae-bf16.safetensors", "vae"),
    ("comfyicu/LTX-2.5", "latent_upscale_models/ltx-2.5-latent-spatial-upscaler-x2-bf16-1.0.safetensors", "latent_upscale_models"),
]

'''
Link scratch weights into the model folders expected by ComfyUI.
透過 symlink 將暫存權重連到 ComfyUI 預期的模型資料夾。
'''
for repo, filename, folder in model_files:
    target = COMFY_DIR / "models" / folder / Path(filename).name
    stored = MODEL_STORE / folder / filename
    target.parent.mkdir(parents=True, exist_ok=True)
    stored.parent.mkdir(parents=True, exist_ok=True)

    '''
    Move previously downloaded weights to scratch without redownloading.
    將先前完成下載的權重搬到暫存區，避免重複下載及佔用保存配額。
    '''
    if target.is_file() and not target.is_symlink() and target.stat().st_size > 1_000_000:
        if not stored.exists():
            shutil.move(str(target), str(stored))
        elif target.stat().st_size == stored.stat().st_size:
            target.unlink()
        else:
            print("Existing model retained:", target.name)
            continue

    '''
    Download only when the scratch copy is absent.
    只有暫存權重不存在時，才執行模型下載。
    '''
    if not stored.is_file() or stored.stat().st_size <= 1_000_000:
        stored = Path(hf_hub_download(repo_id=repo, filename=filename, local_dir=str(MODEL_STORE / folder)))

    '''
    Replace stale symlinks; never duplicate the large model file.
    更新可能失效的 symlink，不另外複製大型權重。
    '''
    if target.is_symlink() or target.is_file():
        target.unlink()
    target.symlink_to(stored)
    print("Model:", target.name, "->", stored)

'''
Expose the same GGUF file to independent GPU loader without duplicating 15 GB of weights.
用 symlink 提供 independent GPU loader 載入 GGUF，不重複複製 15 GB 權重。
'''
gguf_path = COMFY_DIR / "models/diffusion_models/LTX-2.5-Distilled-Q4_K_M.gguf"
multigpu_dir = COMFY_DIR / "models/unet_gguf"
multigpu_dir.mkdir(parents=True, exist_ok=True)
multigpu_link = multigpu_dir / gguf_path.name
if multigpu_link.is_symlink() and not multigpu_link.exists():
    multigpu_link.unlink()
if not multigpu_link.exists():
    multigpu_link.symlink_to(gguf_path)

'''
Obtain the exact original community workflow and its nested subgraphs.
取得原本的社群 Workflow，包含其中的巢狀子圖。
'''
workflow_file = hf_hub_download(
    repo_id="realrebelai/LTX-2.5_GGUFs",
    filename="LTX 2.5 GGUF (WORKFLOW).json",
    revision="112436f97aaf99ce13ecb7b7eca7e2f6c128d3ec",
    local_dir=str(SCRATCH_DIR / ".hf_workflow_stage"),
)
ui = json.loads(Path(workflow_file).read_text(encoding="utf-8"))

'''
Substitute model filenames throughout the UI workflow, including subgraphs.
在完整 UI Workflow（含子圖）中替換模型檔名。
'''
replacements = {
    "gguf": gguf_path.name,
    "text_encoder": FLOW_REGISTRY["models"]["text_encoder"]["basename"],
    "video_vae": Path(FLOW_REGISTRY["models"]["video_vae"]["filename"]).name,
    "audio_vae": Path(FLOW_REGISTRY["models"]["audio_vae"]["filename"]).name,
    "upscaler": Path(FLOW_REGISTRY["models"]["spatial_upscaler"]["filename"]).name,
}

def replace_names(value):

    '''
    Keep this transformation recursive because the workflow contains subgraphs.
    Workflow 包含巢狀子圖，因此需遞迴處理名稱。
    '''
    if isinstance(value, dict):
        return {key: replace_names(item) for key, item in value.items()}
    if isinstance(value, list):
        return [replace_names(item) for item in value]
    if not isinstance(value, str):
        return value
    lower = value.lower()
    if lower.endswith(".gguf") and "ltx" in lower:
        return replacements["gguf"]
    if "gemma4" in lower and lower.endswith(".safetensors") and "enhancer" not in lower:
        return replacements["text_encoder"]
    if "video-vae" in lower and lower.endswith(".safetensors"):
        return replacements["video_vae"]
    if "audio-vae" in lower and lower.endswith(".safetensors"):
        return replacements["audio_vae"]
    if "upscaler" in lower and lower.endswith(".safetensors"):
        return replacements["upscaler"]
    return value

'''
Prepare valid input placeholders for the workflow converter.
準備合法的圖片與音訊預設檔，供 Workflow 轉換器使用。
'''
INPUT_DIR = COMFY_DIR / "input"
INPUT_DIR.mkdir(parents=True, exist_ok=True)
Image.new("RGB", (64, 64)).save(INPUT_DIR / "__ltx25_placeholder__.png")
with wave.open(str(INPUT_DIR / "__ltx25_placeholder__.wav"), "wb") as output_audio:
    output_audio.setnchannels(1)
    output_audio.setsampwidth(2)
    output_audio.setframerate(16000)
    output_audio.writeframes(b"\x00\x00" * 16000)

'''
Replace the sample filenames in nested LoadImage and LoadAudio nodes.
將子圖內的範例圖片與音訊名稱改成實際存在的預設檔。
'''
def patch_placeholders(graph):
    for node in graph.get("nodes", []):
        values = node.get("widgets_values")
        if node.get("type") == "LoadImage" and isinstance(values, list) and values:
            values[0] = "__ltx25_placeholder__.png"
        if node.get("type") == "LoadAudio" and isinstance(values, list) and values:
            values[0] = "__ltx25_placeholder__.wav"
    for child in graph.get("definitions", {}).get("subgraphs", []):
        patch_placeholders(child)

'''
Save the patched workflow where the original engine expects it.
將 Workflow 存至原本引擎預期的目錄。
'''
ui = replace_names(ui)
patch_placeholders(ui)
WORKFLOW_DIR = COMFY_DIR / "user/default/workflows/LTX25-Kaggle"
WORKFLOW_DIR.mkdir(parents=True, exist_ok=True)
(WORKFLOW_DIR / "LTX25_T2V_I2V.json").write_text(json.dumps(ui, ensure_ascii=False), encoding="utf-8")
FLOW_REGISTRY_PATH.write_text(json.dumps(FLOW_REGISTRY, ensure_ascii=False), encoding="utf-8")
print("Models and workflow ready.")

'''
Download the two-stage first/last-frame workflow only when missing.
僅在缺少雙階段首尾影格 Workflow 時下載一次。
'''
FLF_WORKFLOW_PATH = WORKFLOW_DIR / 'LTX25_FLF2V.json'
if not FLF_WORKFLOW_PATH.is_file():

    '''
    Fetch the community workflow that guides both base and refinement stages.
    取得兩階段都加入首尾圖片條件的社群 Workflow。
    '''
    flf_url = 'https://raw.githubusercontent.com/UCHIJ/LTX2.5-first_to_last_frame/main/LTX%202.5-first_frame_to_last_frame.json'
    flf_response = requests.get(flf_url, timeout=90)
    flf_response.raise_for_status()
    flf_ui = flf_response.json()

    '''
    Require two image inputs and four keyframe guides before saving.
    儲存前確認包含兩張圖片輸入與四個首尾引導節點。
    '''
    flf_nodes = list(flf_ui.get('nodes', []))
    for subgraph in flf_ui.get('definitions', {}).get('subgraphs', []):
        flf_nodes.extend(subgraph.get('nodes', []))
    flf_images = [node for node in flf_nodes if node.get('type') == 'LoadImage']
    flf_guides = [node for node in flf_nodes if node.get('type') == 'LTXVAddGuide']
    if len(flf_images) < 2 or len(flf_guides) < 4:
        raise RuntimeError('FLF2V Workflow does not contain two images and two-stage guides.')
    patch_placeholders(flf_ui)
    FLF_WORKFLOW_PATH.write_text(json.dumps(flf_ui, ensure_ascii=False), encoding='utf-8')
print('FLF2V workflow ready:', FLF_WORKFLOW_PATH.name)

'''
Install the original workflow conversion and output-handling engine locally.
在 Kaggle 本機建立原版 Workflow 轉換與輸出引擎。
'''
ENGINE_CODE = '\nimport asyncio\nimport io\nimport math\nimport re\nimport wave\nimport json\nimport os\nimport shutil\nimport subprocess\nimport tempfile\nimport threading\nimport time\nimport uuid\nfrom concurrent.futures import ThreadPoolExecutor\nfrom pathlib import Path\n\nimport requests\nfrom PIL import Image\n\nCOMFY_BASE = os.environ.get("COMFY_BASE", "http://127.0.0.1:8188").rstrip("/")\nCOMFY_DIR = Path(os.environ.get("COMFY_DIR", "/kaggle/working/ComfyUI"))\nFLOW_REGISTRY_PATH = Path(os.environ.get("FLOW_REGISTRY_PATH", "/kaggle/working/ltx25_local_flows.json"))\nWORKFLOW_DIR = Path(os.environ.get(\n    "WORKFLOW_DIR",\n    str(COMFY_DIR / "user/default/workflows/LTX25-Kaggle"),\n))\n\nif not FLOW_REGISTRY_PATH.is_file():\n    raise RuntimeError(f"Flow registry missing: {FLOW_REGISTRY_PATH}")\n\nFLOW_REGISTRY = json.loads(FLOW_REGISTRY_PATH.read_text(encoding="utf-8"))\n\nSTACK_SCHEMA_VERSION = int(FLOW_REGISTRY.get("schema_version", 0))\n\nMODEL_FILES = {\n    tier: item["filename"]\n    for tier, item in FLOW_REGISTRY["models"]["transformers"].items()\n}\nPRIMARY_MODEL_TIER = os.environ.get("LTX25_PRIMARY_MODEL_TIER", "q4km").lower()\nFALLBACK_MODEL_TIERS = [\n    x.strip().lower()\n    for x in os.environ.get("LTX25_FALLBACK_MODEL_TIERS", "").split(",")\n    if x.strip()\n]\n\nVAE_TILE_SIZE = int(os.environ.get("LTX25_VAE_TILE_SIZE", "512"))\nVAE_TILE_OVERLAP = int(os.environ.get("LTX25_VAE_TILE_OVERLAP", "64"))\nVAE_TEMPORAL_SIZE = int(os.environ.get("LTX25_VAE_TEMPORAL_SIZE", "32"))\nVAE_TEMPORAL_OVERLAP = int(os.environ.get("LTX25_VAE_TEMPORAL_OVERLAP", "8"))\n\nCOMFY_LOG_PATH = Path(os.environ.get("LTX25_COMFY_LOG", "/kaggle/working/comfyui_ltx25.log"))\nCOMFY_PID_PATH = Path(os.environ.get("LTX25_COMFY_PID", "/kaggle/working/comfyui_ltx25.pid"))\nTEXT_ENCODER = Path(FLOW_REGISTRY["models"]["text_encoder"]["basename"]).name\nVIDEO_VAE = Path(FLOW_REGISTRY["models"]["video_vae"]["filename"]).name\nAUDIO_VAE = Path(FLOW_REGISTRY["models"]["audio_vae"]["filename"]).name\nUPSCALER = Path(FLOW_REGISTRY["models"]["spatial_upscaler"]["filename"]).name\n\nT2V_I2V_WORKFLOW = WORKFLOW_DIR / FLOW_REGISTRY["workflows"]["t2v_i2v"]["local_name"]\nFLF2V_WORKFLOW = WORKFLOW_DIR / "LTX25_FLF2V.json"\n\nIMAGE_EXTENSIONS = {".png", ".jpg", ".jpeg", ".webp"}\nAUDIO_EXTENSIONS = {".wav", ".mp3", ".m4a", ".ogg"}\nMAX_IMAGE_BYTES = 15 * 1024 * 1024\nMAX_AUDIO_BYTES = 32 * 1024 * 1024\n\nJOBS = {}\nJOBS_LOCK = threading.Lock()\nEXECUTOR = ThreadPoolExecutor(max_workers=1)\n\n\n\ndef job_update(job_key, **fields):\n    # job_key is the dictionary key; fields may safely contain "job_id".\n    with JOBS_LOCK:\n        JOBS.setdefault(job_key, {}).update(fields)\n\n\ndef _comfy_http_error(method, path, response):\n    try:\n        body = response.json()\n        detail = json.dumps(body, ensure_ascii=False, indent=2)\n    except Exception:\n        detail = response.text[:12000]\n    raise RuntimeError(\n        f"ComfyUI {method} {path} failed: HTTP {response.status_code}\\n{detail}"\n    )\n\n\ndef comfy_get(path, **kwargs):\n    response = requests.get(f"{COMFY_BASE}/{path.lstrip(\'/\')}", **kwargs)\n    if not response.ok:\n        _comfy_http_error("GET", path, response)\n    return response\n\n\ndef comfy_post(path, **kwargs):\n    response = requests.post(f"{COMFY_BASE}/{path.lstrip(\'/\')}", **kwargs)\n    if not response.ok:\n        _comfy_http_error("POST", path, response)\n    return response\n\n\ndef get_gguf_loader_choices():\n    # Read only the GPU-selectable GGUF loader exposed by this notebook.\n    # 只檢查本 Notebook 註冊的雙 GPU GGUF Loader。\n    name = \'LTX25GGUFLoaderGPU\'\n    try:\n        spec = comfy_get(f\'/object_info/{name}\', timeout=30).json()[name]\n        return True, spec[\'input\'][\'required\'][\'unet_name\'][0], spec\n    except Exception as exc:\n        return False, [], {\'error\': str(exc)}\n\n\nKEEP_MODELS_RESIDENT = False\n\n\ndef free_comfy():\n    try:\n        comfy_post(\n            "/free",\n            json={"unload_models": True, "free_memory": True},\n            timeout=60,\n        )\n    except Exception:\n        pass\n\n\ndef wait_comfy_ready(timeout=120):\n    deadline = time.time() + timeout\n    last = None\n    while time.time() < deadline:\n        try:\n            response = comfy_get("/system_stats", timeout=3)\n            if response.ok:\n                return True\n        except Exception as exc:\n            last = exc\n        time.sleep(1)\n    raise RuntimeError(f"ComfyUI is not ready: {last}")\n\n\ndef validate_dimensions(width, height):\n    width = int(width)\n    height = int(height)\n    if width < 256 or height < 256:\n        raise ValueError("width and height must both be >= 256")\n    if width % 32 != 0 or height % 32 != 0:\n        raise ValueError("width and height must both be divisible by 32")\n    return width, height\n\n\ndef save_upload_bytes(data, suffix, prefix):\n    input_dir = COMFY_DIR / "input"\n    input_dir.mkdir(parents=True, exist_ok=True)\n    name = f"{prefix}_{uuid.uuid4().hex}{suffix.lower()}"\n    path = input_dir / name\n    path.write_bytes(data)\n    return name, path\n\n\ndef ensure_placeholder_image():\n    """\n    ComfyUI validates LoadImage nodes before execution even when the workflow\n    later disables image conditioning. Some downloaded UI workflows contain\n    stale example filenames such as Test__00139_.png.\n\n    Keep one valid tiny placeholder in ComfyUI/input and patch unused LoadImage\n    nodes to it. The workflow\'s use-image switch still controls conditioning.\n    """\n    input_dir = COMFY_DIR / "input"\n    input_dir.mkdir(parents=True, exist_ok=True)\n\n    name = "__ltx25_placeholder__.png"\n    path = input_dir / name\n\n    if not path.is_file():\n        Image.new("RGB", (64, 64), (0, 0, 0)).save(path, format="PNG")\n\n    return name\n\n\ndef ensure_placeholder_audio():\n    """Create a valid one-second silent WAV for unused LoadAudio validation."""\n    input_dir = COMFY_DIR / "input"\n    input_dir.mkdir(parents=True, exist_ok=True)\n    name = "__ltx25_placeholder__.wav"\n    path = input_dir / name\n    if not path.is_file():\n        with wave.open(str(path), "wb") as wf:\n            wf.setnchannels(1)\n            wf.setsampwidth(2)\n            wf.setframerate(16000)\n            wf.writeframes(b"\\x00\\x00" * 16000)\n    return name\n\n\ndef validate_image_bytes(data, suffix):\n    if not data:\n        raise ValueError("Image is empty.")\n    if len(data) > MAX_IMAGE_BYTES:\n        raise ValueError(f"Image exceeds {MAX_IMAGE_BYTES / 2**20:.0f} MiB limit.")\n    if suffix.lower() not in IMAGE_EXTENSIONS:\n        raise ValueError(f"Unsupported image extension: {suffix}")\n    try:\n        with Image.open(io.BytesIO(data)) as image:\n            image.verify()\n    except Exception as exc:\n        raise ValueError("Invalid/corrupted image.") from exc\n\n\ndef ffprobe_audio_bytes(data, suffix):\n    if not data:\n        raise ValueError("Audio is empty.")\n    if len(data) > MAX_AUDIO_BYTES:\n        raise ValueError(f"Audio exceeds {MAX_AUDIO_BYTES / 2**20:.0f} MiB limit.")\n    if suffix.lower() not in AUDIO_EXTENSIONS:\n        raise ValueError(f"Unsupported audio extension: {suffix}")\n\n    with tempfile.NamedTemporaryFile(suffix=suffix, delete=False) as tmp:\n        tmp.write(data)\n        path = tmp.name\n\n    try:\n        result = subprocess.run(\n            [\n                "ffprobe", "-v", "error",\n                "-select_streams", "a:0",\n                "-show_entries", "stream=codec_name,profile:format=duration",\n                "-of", "json",\n                path,\n            ],\n            capture_output=True,\n            text=True,\n            timeout=30,\n        )\n        if result.returncode != 0:\n            raise ValueError("ffprobe could not decode the audio.")\n        payload = json.loads(result.stdout or "{}")\n        streams = payload.get("streams") or []\n        if not streams:\n            raise ValueError("No audio stream found.")\n        duration = float((payload.get("format") or {}).get("duration") or 0)\n        if duration <= 0:\n            raise ValueError("Could not determine audio duration.")\n        return {\n            "codec": streams[0].get("codec_name"),\n            "profile": streams[0].get("profile"),\n            "duration": duration,\n        }\n    finally:\n        try:\n            os.remove(path)\n        except OSError:\n            pass\n\n\ndef _norm_label(value):\n    text = str(value or "").strip().lower().replace("_", " ")\n    text = re.sub(r"[^a-z0-9]+", " ", text)\n    return " ".join(text.split())\n\n\nALIASES = {\n    "prompt": {"prompt", "positive prompt", "prompt positive", "text prompt", "prompt text", "main prompt", "positive text"},\n    "width": {"width", "video width", "output width"},\n    "height": {"height", "video height", "output height"},\n    "seconds": {"duration", "duration in seconds", "seconds", "video duration"},\n    "fps": {"fps", "frame rate", "frames per second"},\n    "seed": {"seed", "noise seed"},\n    "image_strength": {"img strength", "image strength"},\n    "video_vae": {"video vae"},\n    "audio_vae": {"audio vae"},\n    "text_encoder": {"text encoder", "encoder name", "clip name"},\n    "upscaler": {"upscale model", "upscaler", "spatial upscaler"},\n    "text_to_video": {"text to video", "t2v"},\n    "use_image": {"use image input", "use image", "image input"},\n    "use_audio": {"use custom audio", "custom audio", "use audio", "audio input"},\n    "prompt_enhance": {"enhance positive prompt", "prompt enhance", "enhance prompt"},\n    "tile_size": {"tile size"},\n    "tile_overlap": {"tile overlap"},\n    "temporal_size": {"temporal size"},\n    "temporal_overlap": {"temporal overlap"},\n    "model": {"ltx model", "diffusion model", "unet", "unet name", "model checkpoint"},\n}\nALIASES = {k: {_norm_label(x) for x in v} for k, v in ALIASES.items()}\n\n\ndef _matches_field_label(field, label):\n    label = _norm_label(label)\n    if label in ALIASES[field]:\n        return True\n    if field == "prompt" and "prompt" in label:\n        blocked = ("negative", "enhanc", "system", "instruction", "translated", "output")\n        return not any(word in label for word in blocked)\n    return False\n\n\ndef _iter_all_nodes(ui):\n    if not isinstance(ui, dict):\n        return\n    for node in ui.get("nodes", []):\n        if isinstance(node, dict):\n            yield ui, node\n    definitions = ui.get("definitions")\n    subgraphs = definitions.get("subgraphs", []) if isinstance(definitions, dict) else []\n    for subgraph in subgraphs:\n        if isinstance(subgraph, dict):\n            yield from _iter_all_nodes(subgraph)\n\n\ndef _remove_ui_link(owner, link_id):\n    links = owner.get("links")\n    if not isinstance(links, list) or link_id is None:\n        return\n    kept = []\n    for link in links:\n        if isinstance(link, list) and link and link[0] == link_id:\n            continue\n        if isinstance(link, dict) and link.get("id") == link_id:\n            continue\n        kept.append(link)\n    owner["links"] = kept\n\n\ndef _widget_inputs(node):\n    return [x for x in (node.get("inputs") or []) if isinstance(x, dict) and "widget" in x]\n\n\ndef _widget_index(node, input_name):\n    for i, inp in enumerate(_widget_inputs(node)):\n        if inp.get("name") == input_name:\n            return i\n    return None\n\n\ndef _coerce_widget_value(field, value, ui_type):\n    ui_type = str(ui_type or "").upper()\n    if ui_type == "BOOLEAN":\n        return bool(value)\n    if ui_type == "INT":\n        if field == "seconds":\n            return max(1, int(math.ceil(float(value))))\n        return int(round(float(value)))\n    if ui_type == "FLOAT":\n        return float(value)\n    if ui_type in {"STRING", "COMBO"}:\n        return str(value)\n    return value\n\n\ndef _set_exact_widget(owner, node, field, value, stats):\n    values = node.get("widgets_values")\n    if not isinstance(values, list):\n        return\n    for inp in node.get("inputs") or []:\n        if not isinstance(inp, dict):\n            continue\n        label = _norm_label(inp.get("label") or inp.get("name"))\n        if not _matches_field_label(field, label):\n            continue\n        idx = _widget_index(node, inp.get("name"))\n        if idx is None or idx >= len(values):\n            continue\n        # User-requested controls can be linked to helper nodes in a UI workflow\n        # (the current GGUF workflow links width/height to ResolutionSelector).\n        # Disconnect that incoming UI link so the patched widget value is the\n        # value that survives UI -> API conversion.\n        if field in {\n            "prompt", "width", "height", "seconds", "fps", "seed",\n            "image_strength", "text_to_video", "use_image", "use_audio",\n            "prompt_enhance", "tile_size", "tile_overlap",\n            "temporal_size", "temporal_overlap",\n        } and inp.get("link") is not None:\n            link_id = inp.get("link")\n            _remove_ui_link(owner, link_id)\n            inp["link"] = None\n            stats[f"disconnected_{field}"] = stats.get(f"disconnected_{field}", 0) + 1\n        values[idx] = _coerce_widget_value(field, value, inp.get("type"))\n        stats[field] = stats.get(field, 0) + 1\n\n\ndef _patch_primitives(node, values_by_field, stats):\n    values = node.get("widgets_values")\n    if not isinstance(values, list) or not values:\n        return\n    node_type = str(node.get("type") or "")\n    title = _norm_label(node.get("title"))\n    if node_type in {"PrimitiveString", "PrimitiveStringMultiline"} and _matches_field_label("prompt", title):\n        values[0] = values_by_field["prompt"]\n        stats["prompt"] = stats.get("prompt", 0) + 1\n    elif node_type in {"PrimitiveFloat", "PrimitiveInt"} and title in ALIASES["seconds"]:\n        values[0] = int(math.ceil(values_by_field["seconds"])) if node_type == "PrimitiveInt" else float(values_by_field["seconds"])\n        stats["seconds"] = stats.get("seconds", 0) + 1\n    elif node_type in {"PrimitiveFloat", "PrimitiveInt"} and title in ALIASES["fps"]:\n        values[0] = int(values_by_field["fps"]) if node_type == "PrimitiveInt" else float(values_by_field["fps"])\n        stats["fps"] = stats.get("fps", 0) + 1\n    elif node_type == "PrimitiveInt" and title in ALIASES["seed"]:\n        values[0] = int(values_by_field["seed"])\n        stats["seed"] = stats.get("seed", 0) + 1\n\n\ndef patch_workflow(\n    ui, *, mode, prompt, image_name, last_image_name=None, audio_name, model_file,\n    width, height, seconds, fps, seed, image_strength,\n    tile_size, tile_overlap, temporal_size, temporal_overlap,\n):\n    text_mode = mode in {"t2v", "a2v"}\n    use_image = mode in {"i2v", "ia2v", "flf2v"}\n    use_audio = mode in {"a2v", "ia2v"}\n    values_by_field = {\n        "prompt": prompt, "width": width, "height": height, "seconds": seconds,\n        "fps": fps, "seed": seed, "image_strength": image_strength,\n        "video_vae": VIDEO_VAE, "audio_vae": AUDIO_VAE,\n        "text_encoder": TEXT_ENCODER, "upscaler": UPSCALER,\n        "text_to_video": text_mode, "use_image": use_image, "use_audio": use_audio,\n        "prompt_enhance": False, "tile_size": tile_size, "tile_overlap": tile_overlap,\n        "temporal_size": temporal_size, "temporal_overlap": temporal_overlap,\n        "model": model_file,\n    }\n    stats = {}\n\n    for owner, node in _iter_all_nodes(ui):\n        node_type = str(node.get("type") or "")\n        widgets = node.get("widgets_values")\n        _patch_primitives(node, values_by_field, stats)\n\n        if node_type == "LoadImage" and isinstance(widgets, list) and widgets:\n            # FLF2V assigns its ending image by the titled LoadImage node.\n            # FLF2V 依照 LoadImage 標題指定結尾圖片。\n            if mode == "flf2v" and "last" in _norm_label(node.get("title")):\n                widgets[0] = last_image_name\n                stats["last_image"] = stats.get("last_image", 0) + 1\n            else:\n                widgets[0] = image_name\n                stats["first_image"] = stats.get("first_image", 0) + 1\n            stats["load_image"] = stats.get("load_image", 0) + 1\n        elif node_type == "LoadAudio" and isinstance(widgets, list) and widgets:\n            widgets[0] = audio_name\n            stats["load_audio"] = stats.get("load_audio", 0) + 1\n        elif node_type in {"UnetLoaderGGUF", "UNETLoader"} and isinstance(widgets, list) and widgets:\n            widgets[0] = model_file\n            stats["model"] = stats.get("model", 0) + 1\n        elif node_type == "LatentUpscaleModelLoader" and isinstance(widgets, list) and widgets:\n            widgets[0] = UPSCALER\n            stats["upscaler"] = stats.get("upscaler", 0) + 1\n\n        for field, value in values_by_field.items():\n            _set_exact_widget(owner, node, field, value, stats)\n\n    # Contract checks: fail here rather than silently generating with stale defaults.\n    required = ["prompt", "width", "height", "seconds", "fps", "seed", "model"]\n    if mode in {"i2v", "ia2v", "flf2v"}:\n        required.append("load_image")\n    if mode == "flf2v":\n        required += ["first_image", "last_image"]\n    missing = [name for name in required if stats.get(name, 0) == 0]\n    if missing:\n        raise RuntimeError(\n            f"Workflow patch contract failed for mode={mode}: missing controls/nodes {missing}. "\n            f"Patch stats={stats}"\n        )\n    return ui, stats\n\n\n_OBJECT_INFO_CACHE = {"at": 0.0, "data": None}\n\n\ndef get_object_info_all(max_age=60):\n    now = time.time()\n    if _OBJECT_INFO_CACHE["data"] is None or now - _OBJECT_INFO_CACHE["at"] > max_age:\n        _OBJECT_INFO_CACHE["data"] = comfy_get("/object_info", timeout=60).json()\n        _OBJECT_INFO_CACHE["at"] = now\n    return _OBJECT_INFO_CACHE["data"]\n\n\ndef convert_workflow(ui_workflow):\n    response = comfy_post("/workflow/convert", json=ui_workflow, timeout=120)\n    payload = response.json()\n    if not isinstance(payload, dict) or not any(\n        isinstance(v, dict) and "class_type" in v for v in payload.values()\n    ):\n        raise RuntimeError(f"Workflow converter returned unexpected payload: {payload}")\n    return payload\n\n\ndef normalize_nested_random_noise(api_workflow):\n    """Use local Gaussian noise that preserves nested video/audio structure."""\n    converted = 0\n    for node in api_workflow.values():\n        if not isinstance(node, dict):\n            continue\n        if node.get("class_type") == "RandomNoise":\n            node["class_type"] = "LTX25NestedRandomNoise"\n            converted += 1\n    if converted == 0:\n        raise RuntimeError("Converted workflow contains no RandomNoise nodes to normalize.")\n    return converted\n\n\ndef normalize_api_workflow_for_gguf(api_workflow, model_file):\n    # Place GGUF diffusion on GPU 0 and encoders/VAEs on GPU 1.\n    # 將 GGUF Diffusion 指定至 GPU 0，文字編碼器與 VAE 指定至 GPU 1。\n    counts = {\'transformer\': 0, \'text_encoder\': 0, \'vae\': 0, \'upscaler\': 0}\n    for node in api_workflow.values():\n        kind = node.get(\'class_type\')\n        inputs = node.get(\'inputs\', {})\n        if kind in {\'UNETLoader\', \'UnetLoaderGGUF\'}:\n            node[\'class_type\'] = \'LTX25GGUFLoaderGPU\'\n            node[\'inputs\'] = {\'unet_name\': model_file, \'device\': \'cuda:0\'}\n            counts[\'transformer\'] += 1\n        elif kind == \'CLIPLoader\':\n            node[\'class_type\'] = \'LTX25CLIPLoaderGPU\'\n            node[\'inputs\'] = {**inputs, \'device\': \'cuda:1\'}\n            counts[\'text_encoder\'] += 1\n        elif kind == \'VAELoader\':\n            node[\'class_type\'] = \'LTX25VAELoaderGPU\'\n            node[\'inputs\'] = {**inputs, \'device\': \'cuda:1\'}\n            counts[\'vae\'] += 1\n        elif kind == \'LatentUpscaleModelLoader\':\n            node[\'class_type\'] = \'LTX25UpscalerLoaderGPU\'\n            node[\'inputs\'] = {**inputs, \'device\': \'cuda:1\'}\n            counts[\'upscaler\'] += 1\n    # Reject a workflow that accidentally bypasses either GPU or the two-stage upscaler.\n    # 若 Workflow 略過任一 GPU 或缺少雙階段 Upscaler，直接拒絕執行。\n    if not all(counts.values()):\n        raise RuntimeError(f\'Missing dual-GPU component loaders in converted workflow: {counts}\')\n    print(\'Independent GPU loader mapping:\', counts)\n    return sum(counts.values())\n\n\ndef inject_frozen_external_audio(api_workflow, audio_name, seconds):\n    """Inject existing audio into both stages of the distilled AV workflow.\n\n    Local-only path:\n      LoadAudio -> TrimAudioDuration -> VAEEncodeAudio\n      -> LTXVSetAudioRefTokens -> frozen_audio\n      -> every active LTXVConcatAVLatent audio input\n\n    The normal positive/negative conditioning graph is left unchanged; only the\n    node\'s frozen_audio output is used. For audio-driven modes, both samplers are\n    switched to plain Euler.\n    """\n    if not audio_name:\n        raise RuntimeError("External audio injection requires audio_name.")\n\n    audio_vae_loader = None\n    conditioning_node = None\n    for node_id, node in api_workflow.items():\n        if not isinstance(node, dict):\n            continue\n        if node.get("class_type") == "VAELoader" and (node.get("inputs") or {}).get("vae_name") == AUDIO_VAE:\n            audio_vae_loader = str(node_id)\n        if conditioning_node is None and node.get("class_type") == "LTXVConditioning":\n            conditioning_node = str(node_id)\n\n    if audio_vae_loader is None:\n        raise RuntimeError(f"Audio VAELoader not found for {AUDIO_VAE!r}.")\n    if conditioning_node is None:\n        raise RuntimeError("LTXVConditioning not found in converted workflow.")\n\n    def fresh_id(base):\n        candidate = base\n        n = 1\n        while candidate in api_workflow:\n            n += 1\n            candidate = f"{base}_{n}"\n        return candidate\n\n    load_id = fresh_id("ltx25_external_audio")\n    trim_id = fresh_id("ltx25_trim_audio")\n    encode_id = fresh_id("ltx25_encode_audio")\n    freeze_id = fresh_id("ltx25_freeze_audio")\n\n    api_workflow[load_id] = {\n        "class_type": "LoadAudio",\n        "inputs": {"audio": audio_name},\n        "_meta": {"title": "LTX25 External Audio"},\n    }\n    api_workflow[trim_id] = {\n        "class_type": "TrimAudioDuration",\n        "inputs": {\n            "audio": [load_id, 0],\n            "start_index": 0.0,\n            "duration": float(seconds),\n        },\n        "_meta": {"title": "LTX25 Trim External Audio"},\n    }\n    api_workflow[encode_id] = {\n        "class_type": "VAEEncodeAudio",\n        "inputs": {\n            "audio": [trim_id, 0],\n            "vae": [audio_vae_loader, 0],\n        },\n        "_meta": {"title": "LTX25 Encode External Audio"},\n    }\n    api_workflow[freeze_id] = {\n        "class_type": "LTXVSetAudioRefTokens",\n        "inputs": {\n            "positive": [conditioning_node, 0],\n            "negative": [conditioning_node, 1],\n            "audio_latent": [encode_id, 0],\n        },\n        "_meta": {"title": "LTX25 Freeze External Audio"},\n    }\n\n    concat_count = 0\n    sampler_count = 0\n    for node in api_workflow.values():\n        if not isinstance(node, dict):\n            continue\n        class_type = node.get("class_type")\n        inputs = node.get("inputs")\n        if not isinstance(inputs, dict):\n            continue\n        if class_type == "LTXVConcatAVLatent":\n            # Official distilled A2V keeps the same encoded audio frozen in both stages.\n            inputs["audio_latent"] = [freeze_id, 2]\n            concat_count += 1\n        elif class_type == "KSamplerSelect":\n            inputs["sampler_name"] = "euler"\n            sampler_count += 1\n\n    if concat_count < 2:\n        raise RuntimeError(\n            "Expected a two-stage AV workflow with at least two LTXVConcatAVLatent nodes; "\n            f"found {concat_count}. The pinned workflow may have changed."\n        )\n    if sampler_count < 2:\n        raise RuntimeError(\n            "Expected at least two KSamplerSelect nodes for the two-stage audio path; "\n            f"found {sampler_count}."\n        )\n\n    return {\n        "frozen_audio_concat_nodes": concat_count,\n        "euler_samplers": sampler_count,\n    }\n\n\ndef _is_connection(value):\n    return (\n        isinstance(value, list) and len(value) == 2\n        and isinstance(value[0], (str, int)) and isinstance(value[1], int)\n    )\n\n\ndef validate_api_workflow(api_workflow):\n    """Structural validation against ComfyUI /object_info without queueing GPU work.\\n\\n    Supports normal required inputs plus newer grouped/dynamic inputs\\n    (for example Autogrow values.a / values.b / ...).\\n    """\n    object_info = get_object_info_all()\n    errors = []\n    for node_id, node in api_workflow.items():\n        if not isinstance(node, dict) or "class_type" not in node:\n            continue\n        class_type = node["class_type"]\n        spec = object_info.get(class_type)\n        if not isinstance(spec, dict):\n            errors.append(f"node {node_id}: class_type {class_type!r} is not registered")\n            continue\n        inputs = node.get("inputs") or {}\n        groups = spec.get("input") or {}\n        required = groups.get("required") or {}\n        optional = groups.get("optional") or {}\n        for name in required:\n            if name in inputs:\n                continue\n\n            # Switch branches are lazy/conditional inputs. Only the branch\n            # selected by `switch` is requested by current ComfyUI.\n            if class_type in {"ComfySwitchNode", "ComfySoftSwitchNode"} and name in {"on_false", "on_true"}:\n                switch_value = inputs.get("switch")\n                if isinstance(switch_value, bool):\n                    selected = "on_true" if switch_value else "on_false"\n                    if name != selected:\n                        continue\n                    # If the statically selected branch is missing, fall through\n                    # and report it. A linked/dynamic switch is deferred.\n                else:\n                    continue\n\n            # ComfyUI\'s newer schema supports grouped/dynamic inputs such as\n            # io.Autogrow.Input("values", ...). The converted API workflow does\n            # not contain a literal "values" key; it contains children such as\n            # "values.a", "values.b", ... instead.\n            #\n            # Treat the required group as present when at least one child key\n            # exists. This is generic and is not hard-coded to ComfyMathExpression.\n            dynamic_prefix = f"{name}."\n            has_dynamic_child = any(\n                isinstance(input_name, str) and input_name.startswith(dynamic_prefix)\n                for input_name in inputs\n            )\n            if has_dynamic_child:\n                continue\n\n            errors.append(f"node {node_id} {class_type}: missing required input {name!r}")\n\n        definitions = dict(optional)\n        definitions.update(required)\n        for name, value in inputs.items():\n            # LoadImage/LoadAudio file lists are dynamic. /object_info may have\n            # been cached before a request uploaded its file, so do not use the\n            # cached combo list as our own file-existence validator. ComfyUI\'s\n            # node validation remains authoritative when /prompt is queued.\n            if (class_type == "LoadAudio" and name == "audio") or (class_type == "LoadImage" and name == "image"):\n                continue\n\n            definition = definitions.get(name)\n            if not isinstance(definition, (list, tuple)) or not definition:\n                continue\n            choices = definition[0]\n            if isinstance(choices, list) and not _is_connection(value):\n                if value not in choices:\n                    preview = choices[:20]\n                    errors.append(\n                        f"node {node_id} {class_type}.{name}: {value!r} not in selectable choices {preview!r}"\n                    )\n    if errors:\n        raise RuntimeError("ComfyUI structural preflight failed:\\n" + "\\n".join(errors[:50]))\n    return True\n\n\ndef queue_prompt(api_workflow):\n    response = comfy_post(\n        "/prompt",\n        json={"prompt": api_workflow, "client_id": str(uuid.uuid4())},\n        timeout=120,\n    )\n    payload = response.json()\n    if payload.get("node_errors"):\n        raise RuntimeError(\n            "ComfyUI rejected workflow:\\n"\n            + json.dumps(payload["node_errors"], ensure_ascii=False, indent=2)\n        )\n    prompt_id = payload.get("prompt_id")\n    if not prompt_id:\n        raise RuntimeError(f"ComfyUI did not return prompt_id: {payload}")\n    return prompt_id\n\n\ndef comfy_crash_diagnostics():\n    parts = []\n\n    pid = None\n    try:\n        if COMFY_PID_PATH.is_file():\n            pid = int(COMFY_PID_PATH.read_text(encoding="utf-8").strip())\n            try:\n                os.kill(pid, 0)\n                parts.append(f"ComfyUI PID {pid} still exists.")\n            except OSError:\n                parts.append(f"ComfyUI PID {pid} is no longer running.")\n    except Exception as exc:\n        parts.append(f"PID check failed: {exc}")\n\n    try:\n        meminfo = {}\n        for line in Path("/proc/meminfo").read_text().splitlines():\n            if ":" in line:\n                key, value = line.split(":", 1)\n                meminfo[key] = value.strip()\n        parts.append(\n            "Host memory after failure: "\n            f"MemAvailable={meminfo.get(\'MemAvailable\', \'?\')}, "\n            f"SwapFree={meminfo.get(\'SwapFree\', \'?\')}."\n        )\n    except Exception:\n        pass\n\n    try:\n        gpu = subprocess.run(\n            [\n                "nvidia-smi",\n                "--query-gpu=memory.total,memory.used,memory.free",\n                "--format=csv,noheader,nounits",\n            ],\n            capture_output=True,\n            text=True,\n            timeout=15,\n        )\n        if gpu.stdout.strip():\n            parts.append("GPU MiB total,used,free: " + gpu.stdout.strip())\n    except Exception:\n        pass\n\n    try:\n        if COMFY_LOG_PATH.is_file():\n            tail = COMFY_LOG_PATH.read_text(errors="replace")[-7000:]\n            parts.append("ComfyUI log tail:\\n" + tail)\n    except Exception as exc:\n        parts.append(f"Could not read ComfyUI log: {exc}")\n\n    return "\\n".join(parts)\n\n\ndef _looks_like_connection_loss(exc):\n    text = str(exc).lower()\n    return (\n        "connection refused" in text\n        or "failed to establish a new connection" in text\n        or "max retries exceeded" in text\n    )\n\n\ndef wait_prompt(prompt_id, timeout=3600):\n    deadline = time.time() + timeout\n    while time.time() < deadline:\n        try:\n            payload = comfy_get(\n                f"/history/{prompt_id}",\n                timeout=60,\n            ).json()\n        except Exception as exc:\n            if _looks_like_connection_loss(exc):\n                raise RuntimeError(\n                    "ComfyUI became unreachable during generation. "\n                    "The ComfyUI process probably exited or was killed by memory pressure.\\n"\n                    + comfy_crash_diagnostics()\n                ) from exc\n            raise\n\n        if prompt_id in payload:\n            history = payload[prompt_id]\n            status = history.get("status", {})\n            if status.get("status_str") == "error":\n                # Show only the failing node and traceback, never megabytes of latent tensors.\n                # 只輸出失敗節點與 traceback，避免將大型 latent tensor 列印到畫面。\n                details = [entry[1] for entry in status.get("messages", []) if entry[0] == "execution_error"]\n                if details:\n                    error = details[-1]\n                    raise RuntimeError(\n                        f"ComfyUI node {error.get(\'node_id\')} ({error.get(\'node_type\')}): "\n                        f"{error.get(\'exception_type\')}: {error.get(\'exception_message\')}\\n"\n                        + "".join(error.get(\'traceback\', [])[-12:])\n                    )\n                raise RuntimeError(f"ComfyUI failed: {status.get(\'messages\', [])[-3:]}")\n            return history\n        time.sleep(3)\n    raise TimeoutError(f"Timed out waiting for ComfyUI prompt {prompt_id}")\n\n\ndef extract_output_item(history):\n    # Prefer video-like outputs. Some nodes use "videos", older wrappers use\n    # "gifs", and newer/custom nodes may expose generic "files".\n    video_exts = {".mp4", ".webm", ".mov", ".mkv", ".gif"}\n    fallback = None\n\n    for output in history.get("outputs", {}).values():\n        if not isinstance(output, dict):\n            continue\n        for key in ("videos", "gifs", "images", "files"):\n            items = output.get(key, [])\n            if not isinstance(items, list):\n                continue\n            for item in items:\n                if not isinstance(item, dict) or not item.get("filename"):\n                    continue\n                suffix = Path(item["filename"]).suffix.lower()\n                if key in {"videos", "gifs"} or suffix in video_exts:\n                    return item\n                if fallback is None:\n                    fallback = item\n\n    if fallback is not None:\n        return fallback\n    raise RuntimeError("Generation completed but no output file was found in ComfyUI history.")\n\n\ndef materialize_output(job_id, item):\n    output_dir = Path("/kaggle/working/ltx25_api_outputs")\n    output_dir.mkdir(parents=True, exist_ok=True)\n\n    filename = item.get("filename", f"{job_id}.mp4")\n    suffix = Path(filename).suffix or ".mp4"\n    target = output_dir / f"{job_id}{suffix}"\n\n    response = comfy_get(\n        "/view",\n        params=item,\n        timeout=600,\n    )\n    target.write_bytes(response.content)\n    return target\n\n\ndef mux_original_audio(job_id, video_path, audio_name):\n    """Mux the original uploaded audio back onto the generated video.\n\n    The latent audio drives generation, but the delivered MP4 uses the user\'s\n    original waveform rather than a VAE-decoded reconstruction.\n    """\n    audio_path = COMFY_DIR / "input" / audio_name\n    if not audio_path.is_file():\n        raise RuntimeError(f"Uploaded audio disappeared: {audio_path}")\n\n    output_dir = Path("/kaggle/working/ltx25_api_outputs")\n    target = output_dir / f"{job_id}_with_input_audio.mp4"\n    cmd = [\n        "ffmpeg", "-y", "-v", "error",\n        "-i", str(video_path),\n        "-i", str(audio_path),\n        "-map", "0:v:0",\n        "-map", "1:a:0",\n        "-c:v", "copy",\n        "-c:a", "aac",\n        "-b:a", "192k",\n        "-shortest",\n        str(target),\n    ]\n    result = subprocess.run(cmd, capture_output=True, text=True, timeout=600)\n    if result.returncode != 0:\n        raise RuntimeError(f"ffmpeg audio mux failed: {result.stderr[:4000]}")\n    return target\n\n\ndef looks_like_oom(exc):\n    text = str(exc).lower()\n    needles = [\n        "cuda out of memory",\n        "out of memory",\n        "not enough memory",\n        "failed to allocate",\n        "cublas_status_alloc_failed",\n        "cudaerror: out of memory",\n    ]\n    return any(n in text for n in needles)\n\n\ndef choose_workflow(mode):\n    if mode == "flf2v":\n        if not FLF2V_WORKFLOW.is_file():\n            raise RuntimeError(f"FLF2V workflow missing: {FLF2V_WORKFLOW}")\n        return FLF2V_WORKFLOW\n    if mode not in {"t2v", "i2v", "a2v", "ia2v"}:\n        raise RuntimeError(f"Unsupported mode: {mode}")\n    if not T2V_I2V_WORKFLOW.is_file():\n        raise RuntimeError(f"Base workflow missing: {T2V_I2V_WORKFLOW}")\n    return T2V_I2V_WORKFLOW\n\n\ndef prepare_api_workflow(params, tier):\n    mode = params["mode"]\n    model_file = MODEL_FILES[tier]\n    workflow_path = choose_workflow(mode)\n    ui = json.loads(workflow_path.read_text(encoding="utf-8"))\n    ui, patch_stats = patch_workflow(\n        ui,\n        mode=mode,\n        prompt=params["prompt"],\n        image_name=params["image_name"],\n        last_image_name=params.get("last_image_name"),\n        audio_name=params["audio_name"],\n        model_file=model_file,\n        width=params["width"],\n        height=params["height"],\n        seconds=params["seconds"],\n        fps=params["fps"],\n        seed=params["seed"],\n        image_strength=params["image_strength"],\n        tile_size=params["vae_tile_size"],\n        tile_overlap=params["vae_tile_overlap"],\n        temporal_size=params["vae_temporal_size"],\n        temporal_overlap=params["vae_temporal_overlap"],\n    )\n    api_workflow = convert_workflow(ui)\n    nested_noise_nodes = normalize_nested_random_noise(api_workflow)\n    patch_stats["nested_random_noise"] = nested_noise_nodes\n    normalized_loaders = normalize_api_workflow_for_gguf(api_workflow, model_file)\n    if mode == "flf2v":\n        # Both stages must condition on the opening and closing frames.\n        # 兩個階段都必須加入起始與結尾影格條件。\n        guides = [node for node in api_workflow.values() if node.get("class_type") == "LTXVAddGuide"]\n        frame_ids = [node.get("inputs", {}).get("frame_idx") for node in guides]\n        if frame_ids.count(0) < 2 or frame_ids.count(-1) < 2:\n            raise RuntimeError(f"FLF2V requires first/last guides in both stages: {frame_ids}")\n        image_names = {node.get("inputs", {}).get("image") for node in api_workflow.values()\n                       if node.get("class_type") == "LoadImage"}\n        if not {params["image_name"], params["last_image_name"]}.issubset(image_names):\n            raise RuntimeError(f"FLF2V did not bind both images: {image_names}")\n    if normalized_loaders:\n        patch_stats["api_unet_to_gguf"] = normalized_loaders\n    if mode in {"a2v", "ia2v"}:\n        audio_stats = inject_frozen_external_audio(\n            api_workflow, params["audio_name"], params["seconds"]\n        )\n        patch_stats.update(audio_stats)\n    validate_api_workflow(api_workflow)\n    return workflow_path, api_workflow, patch_stats\n\n\ndef run_once(job_id, params, tier):\n    model_file = MODEL_FILES[tier]\n\n    loader_ready, gguf_choices, _ = get_gguf_loader_choices()\n    if not loader_ready:\n        raise RuntimeError("ComfyUI node UnetLoaderGGUF is not registered.")\n    if model_file not in gguf_choices:\n        raise RuntimeError(\n            f"Model exists on disk but UnetLoaderGGUF cannot select {model_file!r}. "\n            f"Choices={gguf_choices}"\n        )\n\n    job_update(job_id, status="preparing", message=f"Patching/validating workflow with {tier}.")\n    workflow_path, api_workflow, patch_stats = prepare_api_workflow(params, tier)\n    job_update(\n        job_id,\n        workflow=str(workflow_path),\n        patch_stats=patch_stats,\n        message=f"Workflow structurally validated with {tier}; queueing ComfyUI.",\n    )\n\n    if not KEEP_MODELS_RESIDENT:\n        free_comfy()\n        time.sleep(1)\n    prompt_id = queue_prompt(api_workflow)\n    job_update(\n        job_id,\n        prompt_id=prompt_id,\n        status="running",\n        message=f"ComfyUI is generating with {tier}.",\n    )\n\n    history = wait_prompt(prompt_id)\n    item = extract_output_item(history)\n    result_path = materialize_output(job_id, item)\n    if params["mode"] in {"a2v", "ia2v"}:\n        result_path = mux_original_audio(job_id, result_path, params["audio_name"])\n    if not KEEP_MODELS_RESIDENT:\n        free_comfy()\n    return result_path\n\n\ndef worker(job_id, params):\n    try:\n        job_update(job_id, status="preparing", message="Preparing local ComfyUI workflow.")\n        # Model policy comes from this Kaggle notebook.\n        configured_chain = [PRIMARY_MODEL_TIER] + [\n            tier for tier in FALLBACK_MODEL_TIERS\n            if tier != PRIMARY_MODEL_TIER\n        ]\n\n        for tier in configured_chain:\n            if tier not in MODEL_FILES:\n                raise RuntimeError(\n                    f"Notebook model policy contains unsupported tier {tier!r}. "\n                    f"Supported: {sorted(MODEL_FILES)}"\n                )\n\n        # Skip configured fallback tiers that have not been downloaded.\n        available_chain = []\n        for tier in configured_chain:\n            model_path = COMFY_DIR / "models/diffusion_models" / MODEL_FILES[tier]\n            if model_path.is_file():\n                available_chain.append(tier)\n\n        if not available_chain:\n            raise RuntimeError(\n                "No configured model tier is downloaded. "\n                f"Notebook policy: {configured_chain}"\n            )\n\n        result = None\n        last_error = None\n        for index, tier in enumerate(available_chain):\n            try:\n                result = run_once(job_id, params, tier)\n                break\n            except Exception as exc:\n                last_error = exc\n                is_last = index == len(available_chain) - 1\n                if is_last or not looks_like_oom(exc):\n                    raise\n\n                next_tier = available_chain[index + 1]\n                free_comfy()\n                job_update(\n                    job_id,\n                    status="retrying",\n                    active_model_tier=next_tier,\n                    message=f"{tier} failed with an OOM-like error; retrying {next_tier}.",\n                    last_tier_error=str(exc),\n                )\n\n        if result is None:\n            raise RuntimeError(f"All model tiers failed. Last error: {last_error}")\n\n        job_update(\n            job_id,\n            status="completed",\n            result_path=str(result),\n            message="Generation completed.",\n        )\n\n    except Exception as exc:\n        if not KEEP_MODELS_RESIDENT:\n            free_comfy()\n        job_update(\n            job_id,\n            status="failed",\n            error=str(exc),\n            message="Generation failed.",\n        )\n\n\ndef _sample_preflight_params(mode):\n    return {\n        "mode": mode,\n        "prompt": "LTX-2.5 structural preflight",\n        "width": 608,\n        "height": 352,\n        "seconds": 3.0,\n        "fps": 24,\n        "seed": 42,\n        "image_strength": 0.7,\n        "vae_tile_size": VAE_TILE_SIZE,\n        "vae_tile_overlap": VAE_TILE_OVERLAP,\n        "vae_temporal_size": VAE_TEMPORAL_SIZE,\n        "vae_temporal_overlap": VAE_TEMPORAL_OVERLAP,\n        "image_name": ensure_placeholder_image(),\n        "audio_name": ensure_placeholder_audio(),\n        "audio_info": None,\n    }\n\n\n_PREFLIGHT_CACHE = {"at": 0.0, "data": None}\n\n\ndef preflight_all_modes(max_age=30):\n    now = time.time()\n    if _PREFLIGHT_CACHE["data"] is not None and now - _PREFLIGHT_CACHE["at"] <= max_age:\n        return _PREFLIGHT_CACHE["data"]\n    result = {}\n    for mode in ("t2v", "i2v", "a2v", "ia2v"):\n        try:\n            workflow_path, _, stats = prepare_api_workflow(\n                _sample_preflight_params(mode), PRIMARY_MODEL_TIER\n            )\n            result[mode] = {\n                "ok": True,\n                "workflow": str(workflow_path),\n                "patch_stats": stats,\n            }\n        except Exception as exc:\n            result[mode] = {\n                "ok": False,\n                "workflow": str(T2V_I2V_WORKFLOW),\n                "error": str(exc),\n            }\n    _PREFLIGHT_CACHE["at"] = now\n    _PREFLIGHT_CACHE["data"] = result\n    return result\n\n\n'

'''
Keep the helper available to subsequent notebook cells without an external service.
將引擎寫入本機檔案，供後續 Cell 使用，不建立外部 API。
'''
ENGINE_PATH = ROOT / "ltx25_kaggle_engine.py"
ENGINE_PATH.write_text(ENGINE_CODE, encoding="utf-8")
print("Original workflow engine + dual-GPU loader patch ready.")

'''
Create the existing LTX-2.5 nested audio/video noise node.
建立原版 LTX-2.5 同步影音 Noise 節點。
'''
LTX25_NOISE_DIR = COMFY_DIR / "custom_nodes/LTX25LocalNoise"
LTX25_NOISE_DIR.mkdir(parents=True, exist_ok=True)
LTX25_NOISE_CODE = 'import numpy as np\nimport torch\nimport comfy.nested_tensor\n\n\ndef _noise_for_tensor(tensor, generator, noise_inds=None):\n    if noise_inds is None:\n        return torch.randn(\n            tensor.size(), dtype=torch.float32, layout=tensor.layout,\n            generator=generator, device="cpu"\n        ).to(dtype=tensor.dtype)\n\n    unique_inds, inverse = np.unique(noise_inds, return_inverse=True)\n    noises = []\n    for i in range(unique_inds[-1] + 1):\n        noise = torch.randn(\n            [1] + list(tensor.size())[1:], dtype=torch.float32,\n            layout=tensor.layout, generator=generator, device="cpu"\n        ).to(dtype=tensor.dtype)\n        if i in unique_inds:\n            noises.append(noise)\n    noises = [noises[i] for i in inverse]\n    return torch.cat(noises, axis=0)\n\n\nclass _LTX25NestedNoise:\n    def __init__(self, seed):\n        self.seed = int(seed)\n\n    def generate_noise(self, input_latent):\n        latent = input_latent["samples"]\n        noise_inds = input_latent.get("batch_index")\n        generator = torch.manual_seed(self.seed)\n\n        if getattr(latent, "is_nested", False):\n            parts = [\n                _noise_for_tensor(part, generator, noise_inds)\n                for part in latent.unbind()\n            ]\n            return comfy.nested_tensor.NestedTensor(parts)\n\n        return _noise_for_tensor(latent, generator, noise_inds)\n\n\nclass LTX25NestedRandomNoise:\n    @classmethod\n    def INPUT_TYPES(cls):\n        return {\n            "required": {\n                "noise_seed": (\n                    "INT",\n                    {\n                        "default": 0,\n                        "min": 0,\n                        "max": 0xFFFFFFFFFFFFFFFF,\n                        "control_after_generate": True,\n                    },\n                )\n            }\n        }\n\n    RETURN_TYPES = ("NOISE",)\n    FUNCTION = "get_noise"\n    CATEGORY = "LTX25/local"\n\n    def get_noise(self, noise_seed):\n        return (_LTX25NestedNoise(noise_seed),)\n\n\nNODE_CLASS_MAPPINGS = {"LTX25NestedRandomNoise": LTX25NestedRandomNoise}\nNODE_DISPLAY_NAME_MAPPINGS = {\n    "LTX25NestedRandomNoise": "LTX25 Nested Random Noise"\n}\n'
(LTX25_NOISE_DIR / "__init__.py").write_text(LTX25_NOISE_CODE, encoding="utf-8")

'''
Install a small standalone device-placement node pack, not the archived package.
安裝獨立的 GPU 裝置選擇節點，不使用封存的套件。
'''
DUAL_NODE_DIR = COMFY_DIR / 'custom_nodes/zz_ltx25_gpu_components'
DUAL_NODE_DIR.mkdir(parents=True, exist_ok=True)
DUAL_NODE_SOURCE = '# Select independent GPUs without archived DisTorch2 or model sharding.\n# 不使用封存的 DisTorch2，以元件方式分配兩張 GPU。\nfrom contextlib import nullcontext\nfrom contextvars import ContextVar\n\n# Access ComfyUI\'s existing model-loading functions.\n# 使用 ComfyUI 原有的模型載入方法。\nimport comfy.model_management as mm\n# Access the original ComfyUI tensor-loading function.\n# 取得 ComfyUI 原本的張量載入函式。\nimport comfy.memory_management as memory_management\n\n# Preserve the original reader for every other model.\n# 其他模型仍沿用原本的讀取方式。\n_original_read_tensor = memory_management.read_tensor_file_slice_into\n\n# Fall back to PyTorch copying for the LTX spatial upscaler only.\n# 僅讓 LTX Spatial Upscaler 改用 PyTorch 複製，避開 AIMDO 傳輸錯誤。\ndef _ltx25_read_tensor(tensor, destination, stream=None, destination2=None):\n    # Locate the checkpoint associated with this tensor, if available.\n    # 取得此張量所屬的權重檔案路徑。\n    tensor_storage = tensor.untyped_storage() if hasattr(tensor, \'untyped_storage\') else None\n    # Check the source metadata attached to an mmap-backed tensor.\n    # 讀取 mmap 張量所附帶的來源資訊。\n    source_path = getattr(tensor_storage, \'_comfy_source_path\', None)\n    # Use the original file handle when source metadata is absent.\n    # 若沒有來源標記，改從原始檔案控制代碼判斷。\n    if source_path is None:\n        # Read the file-slice descriptor used by AIMDO.\n        # 讀取 AIMDO 使用的檔案區段資訊。\n        file_slice = getattr(tensor_storage, \'_comfy_tensor_file_slice\', None)\n        # Derive a filename only when file-slice metadata exists.\n        # 只有在存在檔案區段資訊時，才取得檔案名稱。\n        source_path = getattr(getattr(file_slice, \'file_ref\', None), \'name\', \'\')\n    # Redirect only the known failing LTX-2.5 upscaler weights.\n    # 只對先前出錯的 LTX-2.5 Upscaler 權重採用替代路徑。\n    if \'ltx-2.5-latent-spatial-upscaler\' in str(source_path).lower():\n        # Report the bypass only on its first actual use.\n        # 只在第一次確實使用替代載入路徑時顯示訊息。\n        if not getattr(_ltx25_read_tensor, \'_reported\', False):\n            # Leave one diagnostic line in the ComfyUI log.\n            # 在 ComfyUI Log 保留一行診斷訊息。\n            print(\'[LTX25] Spatial Upscaler: bypassing AIMDO file-to-device transfer\')\n            # Suppress repeated messages for later weight tensors.\n            # 後續張量不再重複印出訊息。\n            _ltx25_read_tensor._reported = True\n        # Let ComfyUI\'s native tensor copy handle the mmap-backed weight.\n        # 交由 ComfyUI 後續的 PyTorch 張量複製流程處理 mmap 權重。\n        return False\n    # Keep AIMDO\'s efficient path for all other models.\n    # 其他模型保留 AIMDO 的低 RAM 載入路徑。\n    return _original_read_tensor(tensor, destination, stream=stream, destination2=destination2)\n\n# Register the targeted fallback before any model is loaded.\n# 在載入任何模型之前，註冊這項局部替代機制。\nmemory_management.read_tensor_file_slice_into = _ltx25_read_tensor\n\nimport folder_paths\nimport torch\nfrom nodes import NODE_CLASS_MAPPINGS as CORE_NODES\n\n# Keep device selection local to a single loader execution.\n# 讓裝置選擇只影響目前執行的 Loader，不改變其他節點的預設行為。\n_active_device = ContextVar(\'ltx25_device\', default=None)\n_original_torch_device = mm.get_torch_device\n_original_text_device = mm.text_encoder_device\n_original_vae_device = mm.vae_device\n\n# Return the selected device only when a loader requests one explicitly.\n# 只有 Loader 明確指定裝置時，才覆寫原本的裝置查詢。\ndef _selected_device(original):\n    selected = _active_device.get()\n    return torch.device(selected) if selected is not None else original()\n\n# Patch the three official device lookup functions without changing memory accounting.\n# 僅覆寫 ComfyUI 三個裝置查詢函式，不改寫記憶體管理程式。\nmm.get_torch_device = lambda: _selected_device(_original_torch_device)\nmm.text_encoder_device = lambda: _selected_device(_original_text_device)\nmm.vae_device = lambda: _selected_device(_original_vae_device)\n\n# Declare the common GPU choices used by every component loader.\n# 統一定義各模型元件使用的 GPU 選項。\nDEVICES = [\'cuda:0\', \'cuda:1\', \'cpu\']\n\n# Convert a new-style ComfyNode output into the legacy tuple format.\n# 將新版 ComfyNode 的 NodeOutput 轉成舊式節點可使用的 tuple。\ndef _as_tuple(result):\n    # Use the wrapped values directly when the node already returns a tuple.\n    # 若原本就回傳 tuple，直接使用，不額外轉換。\n    if isinstance(result, tuple):\n        return result\n    # Newer ComfyUI returns NodeOutput with its values stored in args.\n    # 新版 ComfyUI 的 NodeOutput 會將輸出值保存在 args。\n    if hasattr(result, \'args\'):\n        return tuple(result.args)\n    # Fail early rather than returning an invalid node payload.\n    # 如果輸出格式無法辨識，立即報錯，不讓壞資料流入 sampling。\n    raise TypeError(f\'Unsupported node output type: {type(result).__name__}\')\n\n# Locate an observable load_device field to verify the selected GPU.\n# 查找模型或 patcher 的 load_device，以確認實際裝置。\ndef _load_device(value):\n    # Check the public wrapper and common patcher locations.\n    # 檢查模型本體與常見的 patcher 欄位。\n    for obj in (value, getattr(value, \'patcher\', None), getattr(value, \'model_patcher\', None)):\n        # Skip containers that cannot report a model device.\n        # 忽略無法提供模型裝置的物件。\n        if obj is None:\n            continue\n        # Check explicit inference allocation before other placement fields.\n        # 優先確認模型實際指定的載入裝置。\n        found = getattr(obj, \'load_device\', None)\n        # Return the first explicit device from the patcher.\n        # 回傳第一個有效的載入裝置。\n        if found is not None:\n            return str(found)\n    # Report non-verifiable models rather than inventing a device.\n    # 若無法從物件確認裝置，不推測載入結果。\n    return None\n\n# Use either the legacy FUNCTION API or ComfyNode\'s execute method.\n# 同時支援舊式 FUNCTION 與新版 ComfyNode execute API。\ndef _load_component(name, device, **inputs):\n    # Find the actual core node instead of inheriting incompatible schemas.\n    # 直接取得原始節點，不繼承可能不相容的節點 Schema。\n    node = CORE_NODES.get(name)\n    # Reject missing plugins before sampling or GPU allocation.\n    # 若缺少必要外掛，先報錯，避免開始耗時生成。\n    if node is None:\n        raise RuntimeError(f\'Missing upstream ComfyUI node: {name}\')\n    # Set the target GPU only for this execution.\n    # 將 GPU 指定範圍限制在目前 Loader 的執行過程。\n    token = _active_device.set(device)\n    # Make kernels that use torch.cuda.current_device follow the same GPU.\n    # 讓依賴 torch.cuda.current_device 的 CUDA kernel 也指向正確 GPU。\n    context = torch.cuda.device(device) if device.startswith(\'cuda:\') else nullcontext()\n    try:\n        # Execute original loader code with the selected CUDA device active.\n        # 使用原本的 Loader 程式，同時設定對應 CUDA 裝置。\n        with context:\n            # Prefer the legacy function when the node exposes one.\n            # 如果節點提供舊式 FUNCTION，優先沿用原本載入方法。\n            if hasattr(node, \'FUNCTION\'):\n                # Invoke the original node without passing the custom GPU selector.\n                # 不將自訂 GPU 參數傳入原始 Loader，避免參數不相容。\n                result = getattr(node(), node.FUNCTION)(**inputs)\n            # Handle newer ComfyNode definitions that only provide execute.\n            # 若使用新版 ComfyNode API，改呼叫 execute。\n            elif hasattr(node, \'execute\'):\n                # Use the actual core ComfyNode class method.\n                # 呼叫核心節點提供的新版執行方法。\n                result = node.execute(**inputs)\n            else:\n                # Stop at an incompatible upstream API instead of guessing.\n                # 遇到無法辨識的新版 API 就直接報錯，避免錯誤分配 GPU。\n                raise RuntimeError(f\'Unsupported upstream loader API: {name}\')\n        # Normalize node output and verify its load device if observable.\n        # 整理節點輸出，並在可取得資訊時核對載入裝置。\n        values = _as_tuple(result)\n        # Reject empty output from any model loader.\n        # 拒絕沒有回傳模型的 Loader。\n        if not values:\n            raise RuntimeError(f\'{name} returned no model\')\n        # Compare actual model device with the requested assignment.\n        # 將模型的實際裝置與指定 GPU 比對。\n        actual = _load_device(values[0])\n        # Fail immediately if ComfyUI selected the wrong device.\n        # 若實際裝置與預期不同，立即中止並提示。\n        if actual is not None and actual != device:\n            raise RuntimeError(f\'{name}: requested {device}, actual load_device={actual}\')\n        # Distinguish observed placement from selection-only cases.\n        # 區分已驗證載入位置與僅完成指定裝置的情況。\n        print(f\'[LTX25 dualGPU] {name}: requested={device}, observed={actual or "not exposed"}\')\n        # Return the result in a format compatible with old-style custom nodes.\n        # 使用舊式自訂節點可接收的 tuple 格式回傳。\n        return values\n    finally:\n        # Restore original device behavior for subsequent graph nodes.\n        # 在 Loader 結束後恢復其他節點的預設裝置選擇。\n        _active_device.reset(token)\n\n# Use the core GGUF loader while explicitly selecting GPU 0.\n# 使用 ComfyUI-GGUF 原有 Loader，明確指定 GPU 0。\nclass LTX25GGUFLoaderGPU:\n    # Expose the same model choices that the installed GGUF plugin recognizes.\n    # 使用目前 GGUF 外掛提供的模型清單。\n    @classmethod\n    def INPUT_TYPES(cls):\n        # Declare exact API names required by the existing workflow.\n        # 定義原始 Workflow 需要的 GGUF 模型名稱欄位。\n        return {\'required\': {\'unet_name\': (folder_paths.get_filename_list(\'unet_gguf\'),), \'device\': (DEVICES,)}}\n    RETURN_TYPES = (\'MODEL\',)\n    FUNCTION = \'load\'\n    CATEGORY = \'LTX25/dual_gpu\'\n\n    # Load a quantized LTX transformer through the upstream GGUF node.\n    # 透過 GGUF 外掛載入量化 LTX Transformer。\n    def load(self, unet_name, device):\n        # Reuse the GGUF implementation without any DisTorch2 modifications.\n        # 完全沿用 GGUF Loader，不加入 DisTorch2 的權重修補邏輯。\n        return _load_component(\'UnetLoaderGGUF\', device, unet_name=unet_name)\n\n# Use the standard CLIP loader with a separate GPU allocation.\n# 將 Gemma 文字編碼器分配到另一張 GPU。\nclass LTX25CLIPLoaderGPU:\n    # Keep the original CLIPLoader model-type options.\n    # 保留 ComfyUI 原本的 CLIPLoader 類型選項。\n    @classmethod\n    def INPUT_TYPES(cls):\n        # Extend the original choice list with the two CUDA devices.\n        # 在原本的文字模型選單加入雙 GPU 裝置選擇。\n        return {\'required\': {\'clip_name\': (folder_paths.get_filename_list(\'text_encoders\'),), \'type\': (CORE_NODES[\'CLIPLoader\'].INPUT_TYPES()[\'required\'][\'type\'][0],), \'device\': (DEVICES,)}}\n    RETURN_TYPES = (\'CLIP\',)\n    FUNCTION = \'load\'\n    CATEGORY = \'LTX25/dual_gpu\'\n\n    # Load the selected Gemma encoder without passing a conflicting device keyword.\n    # 不將自訂 device 參數直接傳入原版 CLIPLoader。\n    def load(self, clip_name, type, device):\n        # Leave the core optional device parameter at its default.\n        # 使用原版 Loader 的預設 optional device 參數。\n        return _load_component(\'CLIPLoader\', device, clip_name=clip_name, type=type)\n\n# Use ComfyUI\'s VAE loader for both video and audio weights.\n# Video 與 Audio VAE 都沿用 ComfyUI 的原版模型載入邏輯。\nclass LTX25VAELoaderGPU:\n    # Use the native VAE choice list, including any supported approximations.\n    # 保留 ComfyUI 原本完整的 VAE 清單。\n    @classmethod\n    def INPUT_TYPES(cls):\n        # Add explicit GPU selection to the existing VAE filename input.\n        # 在 VAE 檔名欄位加入指定 GPU 選項。\n        return {\'required\': {\'vae_name\': CORE_NODES[\'VAELoader\'].INPUT_TYPES()[\'required\'][\'vae_name\'], \'device\': (DEVICES,)}}\n    RETURN_TYPES = (\'VAE\',)\n    FUNCTION = \'load\'\n    CATEGORY = \'LTX25/dual_gpu\'\n\n    # Load VAE weights with the core loader in the selected CUDA context.\n    # 在指定 CUDA 裝置中呼叫原版 VAE Loader。\n    def load(self, vae_name, device):\n        # Reuse the core VAE loader for the chosen model.\n        # 透過原版 Loader 讀取選定的 VAE 權重。\n        return _load_component(\'VAELoader\', device, vae_name=vae_name)\n\n# Adapt the new-style LatentUpscaleModelLoader without inheriting ComfyNode.\n# 不繼承新版 ComfyNode，避免 LatentUpscaleModelLoader 缺少 INPUT_TYPES。\nclass LTX25UpscalerLoaderGPU:\n    # Supply a legacy-compatible input schema based on public folder paths.\n    # 使用目前 ComfyUI 支援的模型資料夾建立舊式相容 Schema。\n    @classmethod\n    def INPUT_TYPES(cls):\n        # Allow selection of the official LTX-2.5 spatial upscaler.\n        # 提供原生 LTX-2.5 Spatial Upscaler 模型選項。\n        return {\'required\': {\'model_name\': (folder_paths.get_filename_list(\'latent_upscale_models\'),), \'device\': (DEVICES,)}}\n    RETURN_TYPES = (\'LATENT_UPSCALE_MODEL\',)\n    FUNCTION = \'load\'\n    CATEGORY = \'LTX25/dual_gpu\'\n\n    # Call either the new execute method or old FUNCTION method as available.\n    # 依安裝的 ComfyUI 版本呼叫 execute 或原本的 FUNCTION。\n    def load(self, model_name, device):\n        # Keep the original spatial upscaler model-loading semantics.\n        # 保留原始 LatentUpscaleModelLoader 的模型載入邏輯。\n        return _load_component(\'LatentUpscaleModelLoader\', device, model_name=model_name)\n\n# Register the four standalone model loaders.\n# 註冊四種不依賴 MultiGPU 擴充套件的模型 Loader。\nNODE_CLASS_MAPPINGS = {\n    \'LTX25GGUFLoaderGPU\': LTX25GGUFLoaderGPU,\n    \'LTX25CLIPLoaderGPU\': LTX25CLIPLoaderGPU,\n    \'LTX25VAELoaderGPU\': LTX25VAELoaderGPU,\n    \'LTX25UpscalerLoaderGPU\': LTX25UpscalerLoaderGPU,\n}\n\n# Expose readable names in the local ComfyUI server.\n# 在 ComfyUI 本機服務顯示易讀的節點名稱。\nNODE_DISPLAY_NAME_MAPPINGS = {name: name for name in NODE_CLASS_MAPPINGS}\n'
(DUAL_NODE_DIR / '__init__.py').write_text(DUAL_NODE_SOURCE, encoding='utf-8')
print('Standalone GPU loader + targeted upscaler fallback installed:', DUAL_NODE_DIR)


## 4. 啟動 ComfyUI


In [ ]:
'''
Check and start the persistent ComfyUI service.
檢查並啟動可重複使用的 ComfyUI 服務。
'''
import requests
import time

'''
Check that all required dual-GPU loaders are registered.
確認雙 GPU 模型載入節點都已註冊。
'''
def loaders_ready():

    '''
    Use the ComfyUI node registry as the health check.
    以 ComfyUI 節點清單確認服務可用。
    '''
    try:

        '''
        Verify every required model loader.
        逐一驗證必要的模型載入節點。
        '''
        for name in ('LTX25GGUFLoaderGPU', 'LTX25CLIPLoaderGPU', 'LTX25VAELoaderGPU', 'LTX25UpscalerLoaderGPU'):

            '''
            Query the running ComfyUI server.
            向目前執行的 ComfyUI 查詢節點。
            '''
            response = requests.get(f'http://127.0.0.1:{COMFY_PORT}/object_info/{name}', timeout=3)

            '''
            Reject unhealthy or incompatible servers.
            拒絕無法回應或缺少節點的服務。
            '''
            if not response.ok or name not in response.json():

                '''
                Report a failed health check.
                回報健康檢查失敗。
                '''
                return False

        '''
        All loaders are available.
        全部模型載入節點皆可使用。
        '''
        return True

    except requests.RequestException:

        '''
        A closed port or interrupted server is not ready.
        連線失敗代表服務尚未就緒。
        '''

        '''
        Permit the startup path to recover the service.
        允許啟動流程重新建立服務。
        '''
        return False

'''
Reuse stable ComfyUI model-loader outputs; switch to none if system RAM runs low. Available choices: classic, none.
重用未更動的模型載入節點輸出；系統 RAM 不足時可改成 none。 可選值：classic、none。
'''
NODE_CACHE_MODE = 'classic'

'''
Validate the selected node cache strategy.
檢查節點快取策略是否有效。
'''
assert NODE_CACHE_MODE in ('classic', 'none')

'''
Keep GPU offloading enabled even when loader results are cached.
即使快取模型載入結果，仍保留 GPU Offload 能力。 避免不同影片的中間節點結果累積大量 RAM。
'''
STARTUP_FLAGS = [
    '--reserve-vram', '2',
    '--cache-classic' if NODE_CACHE_MODE == 'classic' else '--cache-none',
    '--fast-disk',
    '--disable-comfy-compiler', '--disable-pinned-memory',
    '--disable-async-offload', '--fp16-unet',
]

'''
Track whether the running service already uses these settings.
記錄目前服務是否已使用相同設定。
'''
CONFIG_ID = ('ltx25-t4x2-loader-cache-v2', NODE_CACHE_MODE)

'''
Check the existing process before starting another one.
啟動新程序前先檢查現有服務。
'''
previous = globals().get('COMFY_PROC')

'''
Reuse a healthy service started with the same configuration.
相同設定且正常運作時，直接沿用目前服務與模型。
'''
reuse_server = (
    previous is not None and previous.poll() is None
    and globals().get('COMFY_CONFIG_ID') == CONFIG_ID
    and not globals().get('LTX_CODE_UPDATED', False)
    and loaders_ready()
)

'''
Only restart to apply changed server settings or recover a dead process.
只有需要更新啟動參數或服務中斷時才重新啟動。
'''
if reuse_server:

    '''
    Report that cached models remain available.
    顯示模型可沿用，避免不必要的重新載入。
    '''
    print('ComfyUI 已在執行，保留模型快取。')
else:

    '''
    Stop the previous notebook-managed server when changing settings.
    變更設定時停止原 Notebook 管理的舊服務。
    '''
    if previous is not None and previous.poll() is None:

        '''
        Request a graceful termination.
        要求舊服務正常結束。
        '''
        previous.terminate()

        '''
        Wait until the old process exits.
        等待舊程序停止。
        '''
        previous.wait(timeout=20)

    '''
    Avoid starting a second server on an occupied port.
    避免在已被其他 ComfyUI 占用的連接埠啟動第二個服務。
    '''
    if loaders_ready():

        '''
        Require manual cleanup when the process is not notebook-managed.
        若現有服務不是 Notebook 管理，先處理連接埠衝突。
        '''
        raise RuntimeError('8188 已有 ComfyUI 服務；請停止舊程序，再執行此 Cell。')

    '''
    Run ComfyUI with dynamic VRAM and disk-backed model offloading.
    啟用 Dynamic VRAM 與磁碟支援的模型 Offload。
    '''
    command = [sys.executable, 'main.py', '--listen', '127.0.0.1', '--port', str(COMFY_PORT), *STARTUP_FLAGS]

    '''
    Isolate PyTorch allocator settings for the server.
    使用獨立的 PyTorch 記憶體配置設定啟動服務。
    '''
    server_env = os.environ.copy()

    '''
    Remove a potentially conflicting legacy allocator setting.
    移除可能衝突的舊版記憶體配置參數。
    '''
    server_env.pop('PYTORCH_CUDA_ALLOC_CONF', None)

    '''
    Select the native allocator used by the existing workflow.
    沿用現有 Workflow 使用的 native allocator。
    '''
    server_env['PYTORCH_ALLOC_CONF'] = 'backend:native'

    '''
    Capture server logs for subsequent crash diagnosis.
    儲存 ComfyUI 日誌，方便後續分析中斷原因。
    '''
    with COMFY_LOG.open('w', encoding='utf-8') as stream:

        '''
        Run ComfyUI as a persistent subprocess.
        以獨立程序常駐啟動 ComfyUI。
        '''
        COMFY_PROC = subprocess.Popen(command, cwd=COMFY_DIR, env=server_env, stdout=stream, stderr=subprocess.STDOUT, start_new_session=True)

    '''
    Wait for all required model loader nodes to be registered.
    等待全部雙 GPU 載入節點註冊完成。
    '''
    for _ in range(180):

        '''
        Detect startup failures before timing out.
        在逾時前先檢查服務是否啟動失敗。
        '''
        if COMFY_PROC.poll() is not None:

            '''
            Include recent server logs for diagnosis.
            顯示近期日誌，協助找出啟動失敗原因。
            '''
            raise RuntimeError(COMFY_LOG.read_text(errors='replace')[-6000:])

        '''
        Stop polling as soon as the server is healthy.
        服務就緒後立即結束等待。
        '''
        if loaders_ready():

            '''
            Proceed with the running service.
            接續使用已啟動的服務。
            '''
            break

        '''
        Allow model loader initialization to finish.
        等待模型載入節點完成初始化。
        '''
        time.sleep(2)

    else:

        '''
        The service did not become ready in the expected startup window.
        服務未能在指定檢查範圍內完成啟動。
        '''

        '''
        Provide recent logs instead of silently failing.
        顯示最近日誌以便排查。
        '''
        raise TimeoutError(COMFY_LOG.read_text(errors='replace')[-6000:])

    '''
    Remember this configuration so rerunning the cell will not restart it.
    記錄設定，重跑此 Cell 時不重新啟動服務。
    '''
    COMFY_CONFIG_ID = CONFIG_ID

    '''
    Report the effective startup policy.
    顯示目前生效的記憶體策略。
    '''
    print(f'ComfyUI 已啟動：Dynamic VRAM / cache-{NODE_CACHE_MODE}（節點）/ GPU 視顯存情況保留模型。')

'''
Confirm the two-GPU node configuration is available.
確認雙 GPU 節點設定可用。
'''
print('雙 GPU 模型載入節點：正常')


## 5. 共用影片參數


In [ ]:
'''
Set video dimensions and duration for all three modes.
設定三種模式共用的影片尺寸與長度。
'''
SECONDS = 10

'''
Set the video frame rate.
設定影片影格率。
'''
FPS = 24

'''
Set the generation width and height.
設定影片生成的寬度與高度。
'''
WIDTH, HEIGHT = 288, 512

'''
Set a fixed seed, or -1 to randomize every generation.
設定固定種子；-1 代表每次生成都重新抽取種子。
'''
SEED = 42

'''
Set image guidance strength for image-conditioned modes.
設定圖片生成模式的引導強度。
'''
IMAGE_STRENGTH = 0.7

'''
Use the previously tested Q4_K_M model.
使用先前已測試的 Q4_K_M 模型。
'''
MODEL_TIER = 'q4km' 

'''
Preserve tiled VAE decoding for limited GPU memory.
保留適合有限 GPU 記憶體的 VAE 分塊解碼設定。
'''
VAE_TILE_SIZE, VAE_TILE_OVERLAP = 256, 32

'''
Set temporal tile dimensions for VAE decoding.
設定 VAE 解碼的時間分塊大小。
'''
VAE_TEMPORAL_SIZE, VAE_TEMPORAL_OVERLAP = 16, 4

'''
Allow filenames to be retained when uploading local images.
上傳本機圖片時，保留使用者選取的原始檔名。
'''
def save_selected_image(change, folder):

    '''
    Ignore the empty upload event.
    忽略尚未選取檔案的事件。
    '''
    if not change['new']:

        '''
        Keep the existing selection unchanged.
        保留目前已選取的檔案。
        '''
        return None

    '''
    Read either the old or new ipywidgets upload format.
    讀取新舊版本 ipywidgets 的上傳資料。
    '''
    uploaded = change['new']

    '''
    Normalize the uploaded item format.
    將上傳項目整理成統一格式。
    '''
    if isinstance(uploaded, dict):

        '''
        Extract the original filename and its content.
        取得原始檔名與檔案內容。
        '''
        filename, item = next(iter(uploaded.items()))
    else:

        '''
        Read the selected file from the newer widget format.
        從新版 widget 格式取得所選檔案。
        '''
        item = uploaded[0]
        filename = item['name']

    '''
    Keep the original basename without directory components.
    保留原始檔名，排除不必要的目錄路徑。
    '''
    filename = Path(filename.replace('\\', '/')).name

    '''
    Save different image roles in separate folders to prevent collisions.
    依圖片用途分資料夾儲存，避免同名檔案互相覆蓋。
    '''
    destination = ROOT / 'ltx25_images' / folder / filename

    '''
    Create the destination folder on first upload.
    第一次上傳時建立對應的資料夾。
    '''
    destination.parent.mkdir(parents=True, exist_ok=True)

    '''
    Persist the uploaded image using its original name.
    使用原始檔名儲存上傳的圖片。
    '''
    destination.write_bytes(bytes(item['content']))

    '''
    Show the selected image's location for the current mode.
    顯示目前模式所使用的圖片路徑。
    '''
    print('已選取：', destination)

    '''
    Return the saved image path to the mode callback.
    將圖片路徑交回目前模式的上傳處理函式。
    '''
    return str(destination)

'''
Explain that only the selected mode cell determines the next generation.
說明下一段影片只會使用最後執行的模式 Cell。
'''
print('影片設定完成；接著只執行 T2V、I2V、FLF2V 其中一格。')

## 6. T2V：文字生成影片


In [ ]:
'''
Define the prompt specifically for text-to-video generation.
設定只供文字生成影片使用的提示詞。
'''
T2V_PROMPT = '''
電影感多視角鏡頭序列，總時長 10 秒。主角始終是同一位烏黑直長髮、氣質溫柔的韓國年輕女大學生，身穿米白色寬鬆針織開襟衫，肩背帆布托特包；人物臉部特徵、服裝與髮型保持一致。
場景是清晨的櫻花林蔭大道，兩側盛開粉紅櫻花，淡淡晨霧籠罩，柔和金色陽光穿過樹梢，花瓣隨微風緩緩飄落。
0至3秒（側身遠景）：她靜靜站在櫻花大道旁，微微側身眺望遠方，鏡頭平穩取景。
3至6秒（正面中景）：鏡頭流暢移動到她的正前方並緩緩推近，她轉向鏡頭，神情自然柔和。
6至8秒（臉部特寫）：鏡頭繼續推近，她輕輕抬眼直視鏡頭，露出甜美而溫暖的微笑。
8至10秒（微笑與眨眼）：維持正面特寫與眼神接觸，她在最後一秒俏皮地單眼眨眼（wink），隨即保持微笑直到最後一格；微風吹動髮絲。
35mm 底片美學、連貫自然運鏡、浪漫淺景深、柔和粉彩色調、自然皮膚紋理與流暢動作。
'''.strip()

'''
Give text-to-video an explicit clip name because it has no source image.
T2V 沒有來源圖片，因此使用自訂影片名稱。
'''
T2V_RUN_NAME = 'clip01'

'''
Activate T2V without requiring an image upload.
啟用 T2V，不需要上傳任何圖片。
'''
ACTIVE_JOB = {'mode': 't2v', 'prompt': T2V_PROMPT, 'run_name': T2V_RUN_NAME}

'''
Confirm which mode the generation cell will use.
確認生成 Cell 將使用的模式。
'''
print('目前模式：T2V | 輸出名稱：', T2V_RUN_NAME)

## 7. I2V：上傳一張圖片


In [ ]:
'''
Define the prompt specifically for image-to-video generation.
設定只供單張圖片生成影片使用的提示詞。
'''
I2V_PROMPT = '''
The person in the photograph begins to walk naturally, smooth cinematic camera movement.
'''.strip()

'''
Optionally reuse a photo from Kaggle working or an attached dataset.
可直接使用 Kaggle working 或附加資料集中的既有照片。
'''
I2V_IMAGE_PATH = globals().get('I2V_IMAGE_PATH')

'''
Activate I2V while retaining its previously uploaded photo, if any.
啟用 I2V，並保留先前上傳的照片路徑。
'''
ACTIVE_JOB = {'mode': 'i2v', 'prompt': I2V_PROMPT, 'first_path': I2V_IMAGE_PATH}

'''
Keep this mode's selection separate from later mode changes.
將本模式的上傳狀態與後續模式切換隔離。
'''
i2v_job = ACTIVE_JOB

'''
Update this I2V job when a local photo is selected.
選取本機照片後，更新此 I2V 工作設定。
'''
def on_i2v_upload(change):

    '''
    Update the reusable I2V image path.
    更新下次也可沿用的 I2V 圖片路徑。
    '''
    global I2V_IMAGE_PATH

    '''
    Save the image while preserving its original filename.
    保留原始檔名並儲存圖片。
    '''
    selected = save_selected_image(change, 'i2v')

    '''
    Update the selected I2V job only when a file was uploaded.
    只有成功選取圖片時才更新 I2V 工作設定。
    '''
    if selected:

        '''
        Store the image for the next I2V run.
        儲存供下一次 I2V 使用的圖片路徑。
        '''
        I2V_IMAGE_PATH = selected
        i2v_job['first_path'] = selected

        '''
        Show the clip name derived from the input image filename.
        顯示由來源圖片檔名產生的影片名稱。
        '''
        print('影片名稱：', Path(selected).stem + '.mp4')

'''
Provide an upload picker inside the I2V mode cell.
直接在 I2V 模式 Cell 顯示本機照片上傳按鈕。
'''
i2v_upload = widgets.FileUpload(accept='.png,.jpg,.jpeg,.webp', multiple=False, description='選擇 I2V 照片')

'''
Handle future uploads without rerunning this cell.
不必重新執行 Cell，也能處理後續選取的照片。
'''
i2v_upload.observe(on_i2v_upload, names='value')

'''
Display the control for selecting a local image.
顯示本機圖片選取按鈕。
'''
display(i2v_upload)

'''
Show the active mode and current image if available.
顯示目前模式與已指定的圖片。
'''
print('目前模式：I2V | 圖片：', I2V_IMAGE_PATH or '請上傳照片')

## 8. FLF2V：上傳開頭與結尾圖片


In [ ]:
'''
Define the prompt specifically for first/last-frame generation.
設定只供首尾影格影片生成使用的提示詞。
'''
FLF2V_PROMPT = '''
Smooth cinematic motion transitioning naturally between the opening image and ending image.
'''.strip()

'''
Reuse prior selections or enter existing Kaggle image paths here.
沿用先前選取的圖片，或在此填入 Kaggle 中既有的圖片路徑。
'''
FLF2V_FIRST_IMAGE_PATH = globals().get('FLF2V_FIRST_IMAGE_PATH')
FLF2V_LAST_IMAGE_PATH = globals().get('FLF2V_LAST_IMAGE_PATH')

'''
Activate FLF2V with two independent endpoint images.
啟用 FLF2V，分別使用開頭與結尾圖片。
'''
ACTIVE_JOB = {
    'mode': 'flf2v',
    'prompt': FLF2V_PROMPT,
    'first_path': FLF2V_FIRST_IMAGE_PATH,
    'last_path': FLF2V_LAST_IMAGE_PATH,
}

'''
Keep the selected FLF2V job separate from the other modes.
讓 FLF2V 的檔案選取狀態與其他模式互不影響。
'''
flf2v_job = ACTIVE_JOB

'''
Save the uploaded image under its correct endpoint role.
依開頭或結尾用途儲存選取的圖片。
'''
def on_flf2v_upload(change, role):

    '''
    Update the reusable first and last image paths.
    更新可供下次使用的首尾圖片路徑。
    '''
    global FLF2V_FIRST_IMAGE_PATH, FLF2V_LAST_IMAGE_PATH

    '''
    Save with the original image name in a separate endpoint folder.
    依用途分資料夾保留所選圖片的原始檔名。
    '''
    selected = save_selected_image(change, 'flf_first' if role == 'first' else 'flf_last')

    '''
    Keep the prior selection if no file was uploaded.
    沒有選取檔案時保留原本的圖片路徑。
    '''
    if not selected:

        '''
        Ignore empty upload changes.
        忽略沒有圖片的上傳變更。
        '''
        return

    '''
    Update only the endpoint that the user selected.
    只更新使用者本次選取的起始或結尾圖片。
    '''
    if role == 'first':

        '''
        Save the opening image path separately.
        獨立儲存開頭圖片路徑。
        '''
        FLF2V_FIRST_IMAGE_PATH = selected
        flf2v_job['first_path'] = selected
    else:

        '''
        Save the ending image path separately.
        獨立儲存結尾圖片路徑。
        '''
        FLF2V_LAST_IMAGE_PATH = selected
        flf2v_job['last_path'] = selected

    '''
    Show both filenames so the user can confirm the endpoints.
    顯示兩個檔名，方便確認起點與終點。
    '''
    print('開頭：', Path(flf2v_job['first_path']).name if flf2v_job['first_path'] else '未選取')
    print('結尾：', Path(flf2v_job['last_path']).name if flf2v_job['last_path'] else '未選取')

'''
Create two separate local-file upload buttons.
建立兩個獨立的本機圖片上傳按鈕。
'''
flf_first_upload = widgets.FileUpload(accept='.png,.jpg,.jpeg,.webp', multiple=False, description='選擇開頭照片')
flf_last_upload = widgets.FileUpload(accept='.png,.jpg,.jpeg,.webp', multiple=False, description='選擇結尾照片')

'''
Connect each button to its corresponding endpoint.
將兩個按鈕分別連接到正確的影格位置。
'''
flf_first_upload.observe(lambda change: on_flf2v_upload(change, 'first'), names='value')
flf_last_upload.observe(lambda change: on_flf2v_upload(change, 'last'), names='value')

'''
Display both controls in this FLF2V mode cell.
在 FLF2V 模式 Cell 中同時顯示兩個上傳按鈕。
'''
display(flf_first_upload, flf_last_upload)

'''
Confirm the active mode.
顯示已啟用的生成模式。
'''
print('目前模式：FLF2V')

## 9. 生成影片


In [ ]:
'''
Load the original two-stage LTX-2.5 workflow engine.
載入原本的 LTX-2.5 雙階段 Workflow 引擎。
'''
spec = importlib.util.spec_from_file_location('ltx25_kaggle_engine', ENGINE_PATH)
engine = importlib.util.module_from_spec(spec)
spec.loader.exec_module(engine)

'''
Retain models across completed generations; still allow memory-pressure offloading.
保留跨次生成的模型快取，並允許記憶體壓力下的自動 Offload。
'''
engine.KEEP_MODELS_RESIDENT = True

'''
Read an immutable snapshot of the last executed mode cell.
取得最後執行的模式 Cell 設定，避免生成中途被上傳操作更動。
'''
if 'ACTIVE_JOB' not in globals():

    '''
    Select one mode cell before starting generation.
    開始生成前，必須先執行其中一個模式 Cell。
    '''
    raise ValueError('請先執行 T2V、I2V 或 FLF2V 的模式 Cell。')

'''
Copy the active job to prevent later widget events from changing this run.
複製目前的工作設定，避免後續上傳事件改動本次生成。
'''
job = ACTIVE_JOB.copy()

'''
Read the selected mode and its independent prompt.
讀取目前模式及其專屬 Prompt。
'''
MODE = job['mode']
PROMPT = job['prompt']

'''
Keep a placeholder image for text-only generation.
純文字模式使用預設空白圖片。
'''
image_name = engine.ensure_placeholder_image()
last_image_name = None

'''
Require the opening image for either image-conditioned mode.
只要使用圖片生成模式，就必須先取得開頭照片。
'''
if MODE in ('i2v', 'flf2v'):

    '''
    Get the selected first image path.
    取得選取的起始圖片路徑。
    '''
    first_path = Path(job['first_path']) if job.get('first_path') else None

    '''
    Reject missing images before starting GPU generation.
    在 GPU 開始生成前拒絕缺少圖片的設定。
    '''
    if first_path is None or not first_path.is_file():

        '''
        Instruct the user to upload the opening photo.
        提示使用者先上傳起始照片。
        '''
        raise ValueError(f'{MODE.upper()} 請先在模式 Cell 選取開頭照片。')

    '''
    Derive the I2V output name from the original image filename.
    I2V 使用來源圖片的檔名作為輸出影片名稱。
    '''
    RUN_NAME = first_path.stem

    '''
    Upload the first frame to the ComfyUI input folder.
    將開頭照片放入 ComfyUI 的輸入資料夾。
    '''
    image_name, _ = engine.save_upload_bytes(first_path.read_bytes(), first_path.suffix, RUN_NAME)
else:

    '''
    Use the explicit T2V clip name when no image is supplied.
    T2V 沒有來源圖片，使用自訂的影片名稱。
    '''
    RUN_NAME = job['run_name']

'''
Require a distinct ending image for first/last-frame generation.
首尾影格模式需額外指定結尾照片。
'''
if MODE == 'flf2v':

    '''
    Read the ending image selected by the second upload button.
    讀取第二個按鈕選取的結尾照片。
    '''
    last_path = Path(job['last_path']) if job.get('last_path') else None

    '''
    Reject an unset or missing final image.
    沒有指定或找不到結尾照片時立即報錯。
    '''
    if last_path is None or not last_path.is_file():

        '''
        Explain which input is missing.
        提示缺少結尾照片。
        '''
        raise ValueError('FLF2V 請在模式 Cell 選取結尾照片。')

    '''
    Name the clip using both source filenames.
    以開頭與結尾圖片的檔名組合輸出影片名稱。
    '''
    RUN_NAME = f'{first_path.stem}_to_{last_path.stem}'

    '''
    Upload the final frame independently of the first frame.
    獨立上傳結尾照片，避免覆蓋開頭照片。
    '''
    last_image_name, _ = engine.save_upload_bytes(last_path.read_bytes(), last_path.suffix, f'{RUN_NAME}_last')

'''
Generate a fresh 32-bit seed only when -1 is requested.
只有設定 -1 時才建立新的 32-bit 隨機種子。
'''
import secrets

'''
Resolve one seed per run without overwriting the notebook setting.
每次生成決定一個種子，但不覆寫 Notebook 的原始設定。
'''
RUN_SEED = secrets.randbelow(2**32) if SEED == -1 else int(SEED)

'''
Require a supported seed before building the inference workflow.
在建立推論工作流程前，先檢查種子範圍。
'''
assert 0 <= RUN_SEED < 2**32, 'SEED 必須介於 0 至 4294967295，或設為 -1。'

'''
Show the actual seed so a random run can be reproduced later.
顯示實際使用的種子，方便之後重現隨機生成結果。
'''
print(f'本次生成 seed：{RUN_SEED}')

'''
Construct output locations after the selected mode and filenames are known.
確定模式及檔名後，再建立輸出影片與影格的路徑。
'''
OUTPUT_MP4 = ROOT / f'{RUN_NAME}.mp4'
LAST_FRAME_PATH = ROOT / f'{RUN_NAME}_last_frame.png'

'''
Show the chosen output name before the expensive GPU operation.
在耗時的 GPU 運算前顯示這次的輸出檔名。
'''
print('MODE:', MODE, '| MP4:', OUTPUT_MP4.name)

'''
Preserve the audio batch-size setting required by the original workflow.
保留原始 Workflow 所需的音訊 batch size 設定。
'''
queue_original = engine.queue_prompt

'''
Normalize audio batch size before submitting the prompt.
送出 Prompt 前統一音訊的 batch size。
'''
def queue_with_audio_batch(workflow):

    '''
    Inspect the workflow for audio-latent nodes.
    尋找 Workflow 中的 audio latent 節點。
    '''
    for node in workflow.values():

        '''
        Keep audio batch size compatible with the video output.
        確保音訊 batch size 與影片輸出相容。
        '''
        if node.get('class_type') == 'LTXVEmptyLatentAudio':

            '''
            Use one synchronized audio sequence.
            使用一組同步音訊。
            '''
            node['inputs']['batch_size'] = 1

    '''
    Submit the original workflow after the small compatibility adjustment.
    完成相容性調整後，送出原始 Workflow。
    '''
    return queue_original(workflow)

'''
Apply the audio compatibility adjustment to this run.
將音訊相容性調整套用至本次生成。
'''
engine.queue_prompt = queue_with_audio_batch

'''
Pass the selected settings to the original workflow.
將設定值傳入原本的 Workflow。
'''
params = {
    'mode': MODE,
    'prompt': PROMPT,
    'width': WIDTH,
    'height': HEIGHT,
    'seconds': SECONDS,
    'fps': FPS,
    'seed': RUN_SEED,
    'image_name': image_name,
    'last_image_name': last_image_name,
    'audio_name': engine.ensure_placeholder_audio(),
    'image_strength': IMAGE_STRENGTH,
    'vae_tile_size': VAE_TILE_SIZE,
    'vae_tile_overlap': VAE_TILE_OVERLAP,
    'vae_temporal_size': VAE_TEMPORAL_SIZE,
    'vae_temporal_overlap': VAE_TEMPORAL_OVERLAP,
}

'''
Check host RAM headroom before beginning a long I2V job.
長時間 I2V 生成前先確認主機 RAM 剩餘量。
'''
meminfo = Path('/proc/meminfo').read_text()

'''
Read the currently available memory without using external packages.
不額外安裝套件，直接取得可用記憶體數值。
'''
available_kib = int(next(line.split()[1] for line in meminfo.splitlines() if line.startswith('MemAvailable:')))

'''
Report RAM availability for memory-pressure troubleshooting.
顯示可用 RAM，方便排查記憶體壓力。
'''
print(f'可用 RAM：{available_kib / 1024**2:.1f} GiB', flush=True)

'''
Warn when I2V memory pressure may terminate the server.
當可用記憶體偏低時提醒 I2V 可能造成服務中斷。
'''
if available_kib < 5 * 1024**2:

    '''
    Keep the warning nonfatal because available RAM can change dynamically.
    RAM 使用量會持續變動，因此僅警告而不直接中止。
    '''
    print('注意：可用 RAM 低於 5 GiB，請留意系統記憶體壓力。', flush=True)

'''
Submit the generation and measure the complete elapsed time.
執行生成並測量總耗時。
'''
start_time = time.perf_counter()

'''
Run the complete workflow and show a short error log if it fails.
執行完整 Workflow，失敗時顯示精簡錯誤紀錄。
'''
try:

    '''
    Generate the selected clip with its native two-stage workflow.
    使用原生雙階段 Workflow 生成影片。
    '''
    generated_path = Path(engine.run_once(RUN_NAME, params, MODEL_TIER))
except Exception:

    '''
    Report the server exit code when it is no longer running.
    ComfyUI 已中斷時顯示程序結束碼。
    '''
    print('ComfyUI exit code:', COMFY_PROC.poll() if 'COMFY_PROC' in globals() else 'unknown')

    '''
    Check Linux memory-controller counters when they are available.
    系統支援時讀取 Linux 記憶體控制器計數器。
    '''
    events_path = Path('/sys/fs/cgroup/memory.events')

    '''
    Show OOM kill evidence without assuming the process died from OOM.
    顯示 OOM Kill 訊號，不直接假定中斷原因。
    '''
    if events_path.is_file():

        '''
        Print the counters to distinguish memory kills from other failures.
        顯示計數器，以區分記憶體終止與其他錯誤。
        '''
        print('Cgroup memory events:\n' + events_path.read_text())

    '''
    Print the relevant end of the ComfyUI server log.
    顯示最近的 ComfyUI 日誌內容。
    '''
    print(COMFY_LOG.read_text(errors='replace')[-4000:])

    '''
    Preserve the original error and traceback.
    保留原始錯誤與 traceback。
    '''
    raise

'''
Copy the resulting MP4 into the stable output path.
將生成影片儲存到固定輸出路徑。
'''
shutil.copy2(generated_path, OUTPUT_MP4)

'''
Report the output location and total workflow time.
顯示影片輸出位置與完整生成耗時。
'''
elapsed_seconds = time.perf_counter() - start_time
print('MP4:', OUTPUT_MP4)
print(f'生成耗時：{elapsed_seconds / 60:.2f} 分鐘（{elapsed_seconds:.1f} 秒）')


## 10. 預覽與下載


In [ ]:
'''
Confirm that the generated MP4 exists.
確認生成的 MP4 檔案存在。
'''
assert OUTPUT_MP4.is_file(), '請先執行生成影片 Cell。'

'''
Preview the video using a positional path compatible with Kaggle IPython.
使用 Kaggle IPython 相容的位置參數預覽影片。
'''
display(Video(str(OUTPUT_MP4), embed=True, width=448))

'''
Provide a direct download link.
提供 MP4 下載連結。
'''
display(FileLink(str(OUTPUT_MP4)))


## 11. 擷取最後影格（選用）


In [ ]:
'''
Verify output and FFmpeg tools before extracting the final I2V frame.
擷取 I2V 接續影格前，確認影片已完成且具備 FFmpeg 工具。
'''
assert OUTPUT_MP4.is_file(), '請先成功生成 MP4。'

'''
Avoid a less helpful subprocess failure when FFmpeg is missing.
缺少 FFmpeg 時先明確提示，不直接拋出 subprocess 錯誤。
'''
assert shutil.which('ffmpeg') and shutil.which('ffprobe'), '找不到 ffmpeg／ffprobe；請先安裝。'

'''
Count decoded video frames to identify the final frame.
計算影片影格數，找出最後一個影格。
'''
import json

'''
Read the frame count from the saved MP4.
從已儲存的 MP4 讀取影格總數。
'''
probe = subprocess.run(
    ['ffprobe', '-v', 'error', '-count_frames', '-select_streams', 'v:0',
     '-show_entries', 'stream=nb_read_frames', '-of', 'json', str(OUTPUT_MP4)],
    capture_output=True, text=True, check=True,
)

'''
Convert the frame count to a zero-based index.
將影格總數轉成從零開始的最後影格索引。
'''
last_index = int(json.loads(probe.stdout)['streams'][0]['nb_read_frames']) - 1

'''
Save the exact last frame as a PNG for the next I2V clip.
將最後一個影格儲存成 PNG，供下一段 I2V 使用。
'''
subprocess.run(
    ['ffmpeg', '-y', '-loglevel', 'error', '-i', str(OUTPUT_MP4),
     '-vf', rf'select=eq(n\,{last_index})', '-frames:v', '1', str(LAST_FRAME_PATH)],
    check=True,
)

'''
Preview and link the extracted image.
顯示擷取圖片並提供下載連結。
'''
display(Image.open(LAST_FRAME_PATH))
display(FileLink(str(LAST_FRAME_PATH)))

'''
Show the input path to use for the next video.
顯示下一段影片可直接使用的圖片路徑。
'''
print(f"下一段：MODE = 'i2v'，IMAGE_PATH = '{LAST_FRAME_PATH}'")


## 12. ngrok 與遠端生成 API 啟動

In [ ]:
'''
Dependencies are available in the Python standard library.
只使用 Python 標準函式庫處理遠端服務與檔案。
'''
import zlib

import queue
import copy

'''
Use secure randomness when the API requests seed=-1.
API 指定 seed=-1 時，以安全的隨機來源產生實際種子。
'''
import secrets
import io
import json
import os
import shutil
import subprocess
import tarfile
import tempfile
import threading
import time
import urllib.request
import uuid
import zipfile
from email import policy
from email.parser import BytesParser
from getpass import getpass
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer
from pathlib import Path
from urllib.parse import urlsplit, parse_qs

'''
Expose only the inference bridge, never the Jupyter or ComfyUI management interfaces.
ngrok 僅轉送推論橋接 API，不公開 Jupyter 或 ComfyUI 管理介面。
'''
REMOTE_PORT = 8765
REMOTE_SERVICE_ID = '06_LTX-2.5_影片生成_Kaggle'
REMOTE_KIND = 'video'
REMOTE_FIELDS = ['task', 'prompt', 'width', 'height', 'seconds', 'seed', 'image_strength']
REMOTE_IS_LTX = True
REMOTE_SCRIPT = zlib.decompress(b'x\xda\x8dZ[S\x14Y\xb6~\xf7W\xec\x08\x1f\n<R(\xa33\xd1L\xfb\xc0his\x1a\x81\x00\xb4g\x8e\xc7\xa8H\xaavQ\xd9fe\xd6\xc9\xccB9\x1e#\n[\x04\x04\x84V\xc1F\xe8\xf6\xd2\xa2\xb4\nh{\xa3\xb9\xf9c\xa62\xb3\xea\xa9\xff\xc2\xf9\xd6\xdey\xab\x92\x9e\x18\x1fDw\xee\xcbZk\xaf\xf5\xado\xad\xcdA\xd6e(Yf\xe793LuH\xd5\x15\x8d\xd9\x97\x8d\x16\xcbV\x868\xeb\x1a\xf8{K[\xf28\xbbl\x98\x97r\x9aq\x99q\x1dSx\xf2\xc0AV\xdd\x99t\xc6V\x9c\xdb\x0f\xdd\xe5W\xde\x83\x1b\xe1\xd4\xda\xd2b\xed\xc1mw\xfd=\xfb&X\xe4\xec\xcc\xbbwo\xff\xb3|\xfd\x80U\xe4\x19v\x82\xa9\x85\xa2a\xda\x9a:\x98,\xd9\xaa\x96\xa4\xd1t\xce4\n\xe9\x9c\xaa\xf1\xb4fd\x14[5\xf4\xa6\x84f_i;\x9e\xbe\xa4\x0c\raX\x1e\x9d8\xccR\xddg:\xbbS\xe9\xde\x8e\x81\xaf\x9a\x0f\xc8\xd1\xcf\xf7,\x18\xd9\x12\x16\x89]i\xff&\xfa\xabY\x08\x90\xd4\xa017\x93\xfc\n\x8e\x95\xf3\x9a\xe46\xcd\x07\xa0X\x1f\xb7\x15Ug\xf8\xc05\x8b)\x19\xd3\xb0,\x961\nE\x8d\xdb<\xcb\x86\xb8\xceM!\x9f\xf5Wf\xe1(\x8d)\x1aiY\xe0\x05\xc3\x1ci)\x9a\xdc\xb2J&\xcc\x99\xcb\xd1A\xaa>D\xe6\xaa|\xfa\xd1\x9b_\xac~\\u_=\xf6\xee=t\'\xe6`4w\xf5\xb1\xf3\xd3\x94\xf3\xe9\xa53\xbb\xf0\xfb\xcete\xf3\x9936Z]}S]\xfd\xc1\xbd\xf9\xa1\xf6\xf2\x9e\xf3\xf3]\xe7\xd6Res\n\x93\xab\xe3/\x9c\xa9y\xd6#\xb7%k\xfa\x97\xf1u*\xd5\x9b>\xdbs*\xd5\xd5\x9f\xeeK\xf5w\x9eJu\x0f\xc0 \x03f\x89K}p\xbf\x8a\x0e\x03\x15J\xb62\xa8qf\xe9J\xd1\xca\x1b6D\x14\x17\xaf)\x96\xcd\xc8\x1a%R\x90\x14g\x19\xaei$6\x04s\xf6\xee\xbb\xcbego\xday\xf8\xb1\xfax\xda\x17{g\x96\x9d\xc4\x1cV]]s\xd6\x1f@\xf8\xda\xe8\'glF\xaaV\xd9\\\xab\x95\xefU\x9f\xbc\xacl\xder\xae\xbfu\xef\xceTv\x97\xdd\xa5w\x90\x9f\xe4Vs,\xd1qr\xa0\xf3|*\xfd\x9f=\x7fK0\x1d\xa2\xc0\xe2C\x9a1\xa8hVSs\xfb\x01\x86?\x07Y?\xd7x\x06R\xe2~C\xa1\xd8 \xcf\x19\xb0.\xfc\xd3\xb4a\xdb\xd8}$\xfde\xb5\x85)\xe7\xf9\x94\x94\xc4\x99\x9c\x81l\xce\xa7\xb1\xda\xa3\tglB\xaa\xe0\x8c}\x80\x84\x95\xcd\xb2S\x9e\x8a\xe9B\xa2\xd1\x0e\xa6\xa2Z\x9c\x9dW\xb4\x12O\x99\xa6a6%\xaa/\xa7\xc2\xc5l\xa0\xed\xfc?\xcb\xa3\x9dm\xe7\x99;\xb1\xc0Nw\x9d\xc6\xbf\xea\x8d\x82\x8d\x12\xc2\x95N\x1a\xc5\x11aa%c\xab\xc3\x9c}k\x0c2\xdb`p\x91a\xae\xdb\xb0\xbb\xcdMvY\xcd\x0eq\xd8\x9f\x86,F\x0e\xcb2y\x85\xaev\x08kU\x8b\x99%]\x04\xdc\xd3\xf1\xea\xcf\xbb\xde\xd2:t\xc2y\xce\xc7\x15\x18\xb5\xc1\xfc\xb8%okZZ\xbd\xb25U\xd9\xfe\xe0\xde\xfb\rVG\x8c\x86~Gj\x92 \'Xt\x07\xc9\x0c$mj\x0e\xdd\x85d\xb6\x84\xf5\x03\x87P\xf4,S!\x9f\xaagy\x91\xe3/\xc8_\x84\xacE[\xc8\xb6^\x86\xa7H\xd9\xa4!\x9c\xd9[0\xb3\xf3z\xc2y\xf3\x8a\xf5\x8a\x89t0\xb9)N\xc6\xf9\x17\x12\xb4o\xe2\xe2\x81\xde\xbe\x9e\xb3\xbd\x03\xc1\xa0\xdc\x13\xc3\xd8\xf5k\xce\x8bLaEM\xc9\xf0\xbc\xa1!f\xe1\xc4\x04K\xf0\x00f\xf3+v\x8b\xa1k#u\x1ep\x90y\xef\xee\xb9\x0b\xe3\xce\xda})Ge\xf7\x93wo\xb5\xf6h\x0e\x96\xf2~\xd9\xf2\x16w\x9d\xe5\x05or\\\xf8!m\x96\xd6\x95\x02\xe1\x87\x1fL\\\xa7\xe0M\xc7\xceL\x8bi0\x0e\x85I\xbanM7\\S\xda\xec\x7fJ*\x9cR\xa0(\xacCW\x17I\xcaU\x8c\xfb\xa2\xb7d\x0c=\xab\x92\xac\xbeae\x90\xbd\xa8>\x1b\x95\x92J\xe9\xe4MI\r\xc8\x7f_\xbf\x89\\XD$|\xbc\xf6h\xcd\x1b{\x1e\xa8\x92c\xc2\xb4\x08\xa3\xa6\x84\xda6\x0c\x98L\xe4\xb4\x1c\xfe\x11F\xd3\x198Y\xdd\xbd\xe6T\x13q/\xe5,*v>\x88\x1f\xff\x84\xd1M\xbaR\xe0\xce\xfb\x8f\x08\')W\xf5\xe3\x86\xb3\xf7}\x10(b\x834-\x85-z\xf1\xa3I\\a4\x9c\xb8\xd8\xcc \x19F\x93p\xf1\xa6\xf8\x97f\x06t\xe5\xbe\x05\xe5\xb9}\xfc[\n\xf7\x82jY\xa1\x01\xad\xcf\xe2\xfdL\xef\xb9}b\xdeY^\x15_\x1ab\xdf\x9d\xba\xe3\xbd\x9f\xf7v\xb6\x9c\xd7\xdf\xfb\x96\x85B"f\x02% _L\x0f\x84\x1b\x89\x84\\(P)\xfa\x92T-\x91\x9d\x02t\x92\xc7v\xea\x96m\x962\xd2\xb0%\x0b\xb7\x8c\xf0.\x15\xb50\xa5\xfa\xceP\x04\xda\x1a\xc9\xd8Bwv\xce{\xba%\xaf\xbcZ\x1e\xc3\xbd\xca\xa0\x95\xc6\x8e\xee5X\xf1\x19&\xe5\x12W\xe9\xc2\x93\xa5b\x91\x9bM\xcd\xd7\x98\x8fQ\xcb\xabqt\x96\x97\xd8\xe0,\x84MR\x8aS\xdc$X"I\t\xcf\x8c\x92],\xd9L8\xb7\xc0\xa1:V\xe0;4l@\x13\x02\xc3\xd3:\xa9Fe\xef\xa6\xbb5\x17Z\xd9}q\xcf\x99#\xdc\xf7\xaeoUw6\x9d\xf1-g\xf7\r>a\xd0[}\x13\xe8\xd6w\xae;\xdd\xddq\x960!fk\xcb\xe6\x85@\xc6s\x915\xa5\xc3\xe6L\x92\x0fv\xa6\xa1\x93F!7r\xae\x13~O\x92\xe7D\xc0\x86>\xf1z<\xae\xb9{o\x0f\x84%\\A\x8e\x00\xb1\xc6V\xaao\xc7\xdd\x85E\xe7\xe9^\xe8\x11a\x88\x1ff\xe9\x08\x19,e\x98\xa7\xe5\xdd\xa6\x07Gln5\xc5$6y8\xda|\xb8N\x95R.\xa7^9\x1c*\n\xc6\x02\xbf\x0fb\xf2\x9c%\xad\xcf\xaf\x1455\xa3\xda\x94\\XFS\x8b\xf2\x12.\xe7\xb9\x0e?\xf4M\x0f\xdf\xb4p\xdd\x9a\xca\xb3\x81\x8a4\xdd\xfd\xf5\x8e\xbb<\x19\xb7>Q\t\xe9Y\xe3/\xaa\xab\xd7)Y\xfck\xd3\x8b\xb0E\x9a\x11JK\xec\r MaY\x15,G\x87\x8f\x03\xf5\xeb\x81M\xa8\xd9J\xb0\xd8"/\xa5\x1e\x89k\xcf\x16\x9c\xd7{8\xda}\xb4#\x9d\xb2\xb6\\\xae=\x9eq\x9e.\xb8\xd3\xe3\x08A\xef\xfd\x1c&|\x0eb\'N\x04\xe0\xd5\x1eB\x83\xef\x02u2\x84H68\xe2#\x1b\xa1k\x10\x7f\x83%\xdb\x8e\x00\xc2\xcfP\xaf^U\xb6\xa6)\xedOO\xd6&\xe6C\x8ck\x94\x84\xd6\x08\xb8\xff\x0c\xdc\xc2\xd1Fl\x8b>\xfc1\xb4\x81\x85\x95t\x0b8\x0c\xe3\x050\x97\x8b\x82+\x90U\xde\xa8\xb4\x118\x86;\xb9W\xd9\x9cq&^\xc7\xa5t\x17\xaf{\x88\xf8\x99\xb7\xce\xa37\xb5\xe9\x8d\x18\x9aEr7\x80Y\xf8a\x7f,K\xc1\x07\x89\xf8^\xce\xab\x99\xbc\x1fP\xd8\xc1\x97\xf3s\xf0\x92\xa8\xba\x9f\xe5\xf6\'Q\x92)\x11L\xed\x87Q\r\xfbD\x18\xd5-\xa2\x1d\xb7+\xc2\xa2$l6h@;\xcb(\x99\x99\x08\x90\xac\xc0z\x95\xed\x15\x19\xf6\xd5\x89q\xb9k\x03&y\xefo8s\x13\xff\x1e&%\xae6\xc0\xd2\xb5\xb4m\xa4\xafF\xa6\x14c\x89}\xc1\x8a\xeeU\xc6E\x8c4\x81\xaf\xf8\xac;\x86f\x81\xe8\xde\xcc*.Uf\x82\xb8AB~W}v\xb3zw\xaa1\xf9\x87\xde\xfao#W$\x7f=p\xc5\xf4\xf2q\x0b\x16\x08\xecq-M\x9f%\xb3=##\x9d\xf0!\x87\x82\'\xcf\xfe\xd4\xd62\x08\xfc\xb28\xc2Q\xb02\x81]-G\xc9\x85L`\t\x87\xa1\xb2>\xd9\x81s\xcb\x1cL\xdf\xe1\xc8\xee\xe4\x8c\xb3\xbd\x05\xcd\xdd\x85\xd7TJ\xfa{\xd5\x1e\xac\xba\xbf<\xf4V\xd7\x9d\xb59I\xd7\xa8\xc2\xa3(7\xb9mI\x90\xb2\x0c\r\t\x8c\x9c\\\x9c\x8c\\Ht\x19\x9c\xda\xce#\x9b1c\x98\x9b\x97M\xd5\x96\\\x9aS\x18\xf0A\xc3\xb8\x84\xd9\xb6\xedWf\xee\xc6l\xc8\x8d\xdd7[\x10KV\x06\xf2`\x81\xa57\x11\x7f0\xbd\xb3\xf1\x12\x01\xe5\xef@\xa8z\xfb!\x92uD\'\xc8P\xfd\xa9\xd4)X\xde\x172i\x827\x0fr\x94\x87Mm\x87\x0e\xfd\xa9M@\x86\x9cr\x82\x94\x17@\xa1\xeav\x13\x8d5\xd7\xe3.a=\xf4\x85VB5\x9f\x07\r\x96T-\x1b\xa8\xa3\xea9nr\x1dQ\x10\x14\xe8\xc2\xc2\xcb\xab\xbe9o\xafV_.\xc8B\xc1}?\xea\xadN\xf9\xe5\xd0\xd8\x84\xfb\xe2\x89\xfbpE\xaa\xe8m\xdcp\x96gH\x01\xc5\x02\x85\xb1\xd9\x11\xf6\xe5\t\x16*\xf3%\x13\xa2\x83P\x8a\xffJ"Z\xd9\xc6e\xed`fu\xfc-;\xd6\xf6\xc5\xb1/\xfe\xfc\x97\xb6/\x8ecs\x00\x16Q\xed\xeb[\xd0\x8f\x02\x99\x94\xea\xcf\xa3>\xf6\xeb\xa0\x12\xa2B(d\x19P\x92\xec\x03\xa2Aw\x96\x01:\x0er8\x0bJ\x80l)\x83\x19\xa2>\x12\x99\xe4\xf1\x06\x00\xc7\xd9xV{\xb0$3\x1bavp?\xee\xc2o\x95\xbdO\x95\xdf\xa6P\xfd\xd4\xc6g\xbc\xd9=\xdfsd\x99\xfd~\xce\xfdi\x99\xb4+\x9ad\xe9\\"^\x0c\tQ~\xdfyp5P\xf7ZP\xb9\x05\xcc\xce\'EAS\xc2bJ\x8e\xaa\xb6\xfd+\xa4\x10\x8c\x98\x82\xbb\xba\xa4\x1b\x97eM\xf2\x84\xdc*,\x8c$\x0cAZ\xba\x8b\x9b\xbe\xef\xc7\xf1\x08\xc8%\xd3&\x91\x94\x90r\xf7\x9c\x1b\xe8=7\x90>\xdb{\x0c\x0e\xd6\xd7\xd33\xc0Z\xeb\xc23Y(\x1eK\x1c\xe8\xea\xe8\x1fH\x9f\xee\xc3\x80\xe8\x90\xec?UDrZbOQ\x1f\xaa\xbf\xa4L\xde\xb0\x10\xbbq6\xe8;\x9fOUP#\x11m$\xbe\r\x82\x1b%|8^\xb5|\x9f2\x13\x82X\xb0\xf1\xd1)o\xfd>\x9cN^`\xad\xbcH\x86\x97\xc4k|K\x9a!\xba\x99\x04%\xffvT.\xf4\x13\xee\xf6\x7f\x0c\xaa\xd2\xff#\xc5\x93$\x8d\xb8\xa0^\xa0\x0e7}\xf2\xaa\x94\xb2\xaa\xc1\x06\x15;\x93o\xb1\xd4\xff\xe5Ax\x0b\xe4A4\x85\x0c!$\xb3\xf1x\x91\xad\x19\x19\xccQ\xd3\xca\x9d,\x83\xae@\xd8\xda\xc3\xb7\xd5\xd5[rw&v\x8f"\x1e\xb8V\xe2\xe9p\xd7\x10q\xe5\xb8\xaceI\xdan\xc3,(\x1a\xad\x8d\x89*7\x0b\xea\x9b\xd2`A\xb5C\xa0\x8aJ\xebZy\x14\xb6\xf2KhF\xb5\xff\xfb7\x00()\x15\x19:\xda\x8b$\xca\xf2\x1c\x93\xa7\x13\x06\xa6\xc5qi1\xa5)\xd09\xac\x08Q\xbd\x14\xb9_\xbc\x84\x1d>"vbU\x0b\x85\x1f\xaa|\x1d\xcemE\x9cz\nt$\xb2Res\x8dd\x90Z\xf9\x0b\xbc\x8drm\xfb\xa7\xb0H\x14\xec#K`\x15Y}\x98x\x81U\xcf?d\xb5\xdfh\x1ej\xbc\xc1\xc1\xa8mE\n\tY\x87\xd5,7|\xff\x8c\xf3\x12\x84\x19\xeer\x9f\xfb\x12\xe1Dq%\xfc\xce[\xdat\xd6\x7f\x8b\x93\x15\x802\x89(\x89\\\x06\xb1a\xa5\xed\x91"\x07\x93#*\xda5\xf0\xf7\xf3)\x18\x7f\xa4K\xe8\xd7A"&"\xc9#&/\xd2\xd0\x88\x9e\xc9\x9b\x86\x8es\xb3\xbe6\x16\xa5?\x80t\xb2a\x89_6m\x96\x05!\x99v\xd7V\xa4\xe8q\xc9\xe8\x0f\x89v!!\x08\x99\x95\xb8x!!4K\x93f\x89\x8bp\xb9\xa3\x01\xf9\xe8\x17.\xb4\xbf\x9f\xc7Q\x0b\x9e\xa8\x85vU5\xd5\x1eaJ\xf6\xdb\x92e\x17\xa0]x\xd1\xeb\xd3\x04\x9e\xc2Tn\xf9y\xf5\xc5\'w\xfe\x9d\x84,\xe9\x91\r\xf1\x126\xcf\xb8]2uV\x1f\x18\x91\xebQ0t\xa0\x86\x19\x89\xc5\xed\x1f\x89"k\xbc\xa8\x01F\xf5\x9c\xf4\xfa\x06\xa9V\xee\x8bZ\xe7mc\x97k\x9f`\x84\xbd\xf6\x8f\x0e\x81)\xb8\xf8z`\xf7q\xc4\n\xca\xcd}\xf1\x03rIDp\xca; nu]\xf1\xb8y\x8a\n\xf0\xd6\x82\x04W\x85\xa5d\xff\xab]\xa2\x9d\x1c\xf1\x9b_\xedL\xf6\xc4\xfc\xd1\xcbj\x16UE;\xfb\xa6\xf3\xd4\xc0W\xfeX\x9e\xabCy\x9a\xf9U\xaa\xf3\xccW\xc1LY\x00Y\x18\xeeO\x9d\xec\xe9>\xd5\xef\x8f\xe7\x8a4v\xba\xb7?\x9c\xc7\xb3\x18\x08r\x9e?\x1a\xb1G|\x8bQI\xf9\xb5\x81`bJ#\xe5\x94\xf3\xa4M\xfd)\x7f\xdcT\x13\xd3@9\xe3G#\xe3b\x81\xd0\xb5\xf3l\xc7\x99T\xba\x7f\xa0/\xd5}&TzX\xe1i\x9b^\x08\x84\xef\xb7\xb3\xf3\x1d\xa9\xf4@g\x17\xe6u\xfeW\xaaq\x0eQ?M)\xc6\xa7\xf5\x9cO\xf5uu\xf4\xc6gr\xe2\x94\x8aV\xb7c\xealoO_G\xd7g\xbb\x06s\x1bv\x0e\xa6\x87\xbb_\x13\x04"\xcf3\x97\x182\xa9\xcd\x90\x8bY\x1e4\xdb4\xc0t\x02\x1a\xc7a\x18\xd1\x1aR@-\xf0\x83\xfa(TK\x12\xe2\xcf\x7fD\x0e\xad-\xdc\x15\x83a/\x0b\xac\r\x00W}1S\xd9\xdc\x06\xbb\x11\xdb:\x93\xbf\xd4\x9e\xfeP\x1b\x9f%\xff*\xf0\x02\x08\xa1\x11\x94\xab\x89V\xb8S\xa6\xd5\x1fM4K\xaaOm\xd3\x86Fo\xa6d\x9a\xb22Q\x86\x15U\x13O\x04\xf21#\xe4\xd1\xb2\xec\xc2Rn\x92\xf7\x17\x95\xcc%j\xce\x89\xec\xb99#kzg}\xb2\xfa\xf3\x8f\x08\xc6\xca\xf6\x07\xa0\x84\xb7\xf4\xce\xbd\xbd"{\x89\xce\xec\x06Eh\xf0\xaa\xe1\xceo"T\x04\xe1\x0c\x0eL_R\xa9#M4@\'\t5Q\xb7\x14\x81\x08M\xcd\x17\x8e^\x14Y\x84\xc6(\x8b\xf8\x1a\xc9\xcf4\x88D"\n_\xb1\x86\xfa\x84\x16\xc9\r>\xc1\x0b\x1d\xc1\x01\xed\x89\xe6f_mQE\x90\xf5\xfc\xd3%\xee\xd0\t\x8dO8\xb6i\x94\xb0\xd6\xca\x1bFP-\xf8TT(D\x9b\x84\xfc\xd3\xbd}\x07\x8c\xba\xe1\xe5&N;\xe3k\x1e\\\xad\xd7\xbc\x95\x1d=\xd2v\xec\xd0\xa1\xb6\xf6\xe4\xd1\xdc5vF\xfd\x1bhON+Y\xf9\x13\xf4\x8c#\x04\xffF1uYU\x91[\xf8\x17\x14\x8aZP\x00\xa9\xdc\x84e\xa8*\x93 \x06z$\xe8\xb37\xff\xa1\xe1\x06\x9c\xd1\xd9\xca\xeem*\xbdf\xe7j7\xef\x88\x1d1\xa5\xfa\xddn\xad\xfc\x88\xea\xa0\xe5\x19gj\x11\xc9\xdd]\xf8\xe8wf\xea\x05\xfe\x92\x1dg\x87B\xa1\x0f\xc42\xb8 \x13\x10\x95\x1cF7\xf4\x9cb+\xf4h\x93QJ\xc8\x90\x91\x83\x91\xfd\x89\xf1\x8b\xa7\x0e\xce\xb2#@\x0c5\x83\xcc4\x12d \x9a\xe1w\xef\xc7g\xdd\xe5\xef\xdc\xe9Qok\xba\xba~\xcb\x99\x9a\'\xe2\xbc\xf4\xc8]{\xea|7V]{\xe6|\x7f\xabZ\x9e\x86\'J\xa7#\xb9\xd7\x9e\x04\x19\xc9\'\x97\xee\xdbU\xf7\xc6,L\x1f\xdd\x03#\x1b\xa0\x8a9N\x06\xc7\x96\xd5\x97S \x82\x98\xe5\xbd\xdd\x06\xcb\xfa\xfc.?\xbf\x94X\xce\x8d\x9a_\xa2\x14(pE\xfaP\x9e\x87\xaf\x87\xa8\xf7\x94\xa2\x85\xbcb\xab\x05\xd9\xf9\x97Oj\xfe\xa3\xd93w\xf3\x15t\xf56w%\x8d\x16/\xa6\xe4\xcfi\x9a\x8f\xf0\x10\xcb\xc0\xb8s\xe9\x8cQ\xd2mj\xfc\n\x97F\xf5TwL\x94\xf5!\x88E\xbc^\xa1\x1fp{N\xbd\x18 \xce\x10\x05\x0c\x04\xcf\xe1>\xacH\x12\xca\xfa\xf3\xef\xc2\xacEf~\xfa\xc6\x9d\'a\xa4\xe3{\xbf\xeey\xaf\x1f\xd7\xa67\xaa/\x9ez\xef\xca\xb5\xe9\x1b$\xa4m\x8eD\xaf\x0c~c\xa0.\x8f\x8a\xde\x8d\xe0o\xf4\x86\x04\x17\x15m\xe7\xf0\x999\xca\xa6qf\x84,\n\xcb\xec\xf3\x9e\xecC\xa2\xa0t\xc1-\xfb\xd6\xe7\xd9\xba\x96\x9d\x9f\x7f\xa8\xb5i\x80\x815\x05\x05\xd0a&\x93\xb1\xac5\xba\x90\x1eR}\x80\x07~%\xc3\xc1\x11R\xe2\x07.2j?\n\xcc\x88\xa2\nP\x08\xe3e\x88\xd6\x8a\x88TE\xabL7\x04\x96\xcb\x1e\x84\x1e6\xcd\x0e\x86=g\xe7\xe3\xaf2\xa6"{\xa2&\xdf\x9a\xa5\x1a\xf5\xc7\x87\xde\x93\x9d\x06\x9f\r\xd6\x85\xa7Q5t\xb2\xe7\xec\xe9\x7f\xa4\xc1\x10N&\x8b\x86\xa6I\xecKD\xa3\x89\xba\xb7T\xd9bH\x94tQ\x8cFm5\x99\x9f\xbaT\xbdt%\x80=p\x07\x00\x9e\xa6A~\xdf\xbd,\xa9\x1d\xf4\x1e\x11\x05m\x18\xbea\xc3J\x04\x8a{o\xc3\x9d}\x07\x9dd\x93\xd5\xdf5\x82\xfb\xdb\xcf\x9d\x89\x0f\xce"\xe0g\x82\xa0\x7fq5PS\xbey\xd6\xddX\xa2\xd5\x1a\xb1ZsVkf\x08\xe0[l\x95\xb2%\xe5\xccH|Q\xac\xf6\xf4\x9ce\x97\xe8\x01\x9eSE \xda ~\xbe\x02\x91+\x15b\xb5T\x06(\xc9\xb2*=z\xd1\xbb\x06\x16\x86\x8f\xc5\xe2\x1a\xc4V_\xab\xe2=\xfbVu\xf1\xa1x\x8c\xf7\xd1\xc4\x19\x1d\x17=!\xba7"\xbdK\x8fb\xcd\xd6\x98\x06\xfbwU{\xe9&\xfd\xe8\xf4m\n.){\xeaC%\xd5\xca\x07HN\x8a\xf8\xcf\xbf\x86x/\xa4\xd0\x04\x80X\xf1\nGJ\x1b\xda\x91\xc4\xdc^qf\xca\xce\xc4\xcd\xd0\xdc\xde\xfb\xeb@?\xaay\xc6>T\xb6\x17d\xac\xc6k\x8a\xc0\xb9\x84\x81\x83\xe3\xa5"\xed\xff\xad\'\xd8\x7f\xd4i\x15c\r\xa1\xf5#\xa5LD\xf8\xb0\xa2\x8b\xc7\x81\xa0\xb1\x19x\xad\x1f*\x00\x9bzc\xbb\xcb\xe5\xea\xa7\xef\x89\x1d\x073\xdd\xfb+\xd5\x17\xd3\xce\xd8\xf3X]&\xa5\x94n\xdd\xd5s&&\x87\xc00\xebD\xc2\xe4\x82P&\x9a/\xb4\x1c;r\xe4H\xfb\xc5\x98|\xb1\xe6@\xc8\xd9%\xf6\x11$\xda&\xd6\r\x82\xc4\x84\x80\x13k\x01H\x83\xc1~\xd1\xb4\xba\xdf\x0c\xa8{\xd8\xc7A%MT\xed\xd4\x96\x81\xc4\xb2N\xb0\xe4/Z\xf8-\x14\xf9\xb8*\n\x858t97~u\xd6~p&^;K\xd4\x1e\x92\xc5i\xd4\xee\xb1\xf2\xe2\x17Y\xe8e\xbe\xad\xa9\x1e\xde\xe2-\x918\xa9\x11\xea\xd6\xf7\xab\xa4\xbe\x86\x1d\xaf\x05\x83\xec\xe3s\x99Xi\\\xd9\x9d\xf1v\xd7\xc9wD"\x90\xd2F\xc9\xc8O_i\xbf\xce\xd8?#\xb1\x16\x16%\xad\xb0\xb5\xd3\xd8\xc9i\x0eyQ\xfc\x10bF\x8d\x87\xb4\xb2?\x1fiO\xb6\x81\x16\xc1\xcbks?\xfc\xbe3\xd18G\xb2&\xef\xf9\x9d\xdfw&\x81\x11\xff\x0f\x87C\xd0\xfa').decode('utf-8')
REMOTE_ALLOWED_TASKS = ['t2v', 'i2v', 'flf2v']
REMOTE_MAX_IMAGES = 2

'''
Keep job directories isolated between different notebook services.
各 Notebook 使用獨立工作資料夾，避免不同服務互相干擾。
'''
REMOTE_ROOT = Path('/kaggle/working/remote_api_jobs') / REMOTE_SERVICE_ID
REMOTE_ROOT.mkdir(parents=True, exist_ok=True)

'''
Share existing state when the API cell is executed again.
重跑 API Cell 時保留原有工作及鎖定狀態。
'''
REMOTE_JOBS = globals().setdefault('REMOTE_JOBS', {})
REMOTE_JOB_LOCK = globals().setdefault('REMOTE_JOB_LOCK', threading.Lock())
REMOTE_ENGINE_LOCK = globals().setdefault('REMOTE_ENGINE_LOCK', threading.Lock())
REMOTE_QUEUE = globals().setdefault('REMOTE_QUEUE', queue.Queue(maxsize=32))
REMOTE_UPLOAD_SLOTS = globals().setdefault('REMOTE_UPLOAD_SLOTS', threading.BoundedSemaphore(2))

'''
Results remain available for up to 24 hours, subject to an 8 GiB storage limit.
結果最多保留 24 小時；若超過 8 GiB，會提前清除最舊結果。
'''
REMOTE_RESULT_TTL = 24 * 3600
REMOTE_RESULT_BYTES = 8 * 1024**3
REMOTE_MIN_FREE_BYTES = 2 * 1024**3

'''
Do not start a second GPU job when a long-running backend appears stalled.
當底層生成疑似卡住時，停止接收新工作，絕不啟動第二筆 GPU 推論。
'''
REMOTE_STALL_SECONDS = 2 * 3600

'''
Report only checks that can be verified without loading another model.
只檢查不會額外載入模型的服務就緒條件。
'''
def remote_backend_ready():
    if REMOTE_KIND == 'image_multi':
        return bool(globals().get('QWEN21_STATE', {}).get('pipeline'))
    if REMOTE_SERVICE_ID.startswith('06_HeartMuLa_'):
        return globals().get('pipe') is not None
    if REMOTE_SERVICE_ID.startswith('06_YuE2_'):
        gpu = globals().get('GPU_ID', 0)
        process = globals().get('SERVER_PROCS', {}).get(gpu)
        return bool(process and process.poll() is None)
    base = globals().get('BASE')
    if not base and REMOTE_IS_LTX:
        base = f"http://127.0.0.1:{globals().get('COMFY_PORT', 8188)}"
    if not isinstance(base, str):
        return False
    try:
        with urllib.request.urlopen(base.rstrip('/') + '/system_stats', timeout=3) as response:
            return response.status == 200
    except (OSError, ValueError):
        return False

'''
A Python thread cannot safely terminate an already running CUDA inference.
Python thread 無法安全強制終止正在執行的 CUDA 推論。
'''
def remote_stalled():
    with REMOTE_JOB_LOCK:
        return any(record.get('status') == 'running' and
                   time.monotonic() - record.get('_start_clock', time.monotonic()) > REMOTE_STALL_SECONDS
                   for record in REMOTE_JOBS.values())

'''
Remove obsolete input files once the model no longer needs them.
模型不再需要輸入檔案時立即移除，避免參考圖片長期佔用空間。
'''
def remote_remove_inputs(job_id):
    folder = REMOTE_ROOT / job_id
    for file in folder.glob('*'):
        if file.name.startswith('input.') or (file.stem.isdecimal() and file.suffix.lower() in {'.png', '.jpg', '.jpeg', '.webp'}):
            file.unlink(missing_ok=True)

'''
Keep only finished results; never delete active jobs or in-progress downloads.
僅清理已結束工作，絕不刪除排隊、推論或正在下載的檔案。
'''
def remote_cleanup():
    now = time.time()
    with REMOTE_JOB_LOCK:
        finished = [(job_id, record) for job_id, record in REMOTE_JOBS.items()
                    if record.get('status') in ('done', 'error', 'expired') and
                    not record.get('_downloads')]
        finished.sort(key=lambda item: item[1].get('finished_at', now))
        sizes = {}
        for job_id, record in finished:
            folder = REMOTE_ROOT / job_id
            sizes[job_id] = sum(path.stat().st_size for path in folder.glob('**/*') if path.is_file()) if folder.exists() else 0
        total = sum(sizes.values())
        for job_id, record in finished:
            old = now - record.get('finished_at', now) > REMOTE_RESULT_TTL
            if old or total > REMOTE_RESULT_BYTES:
                shutil.rmtree(REMOTE_ROOT / job_id, ignore_errors=True)
                total -= sizes[job_id]
                record.update(status='expired', error='結果已過期或因儲存空間不足而清除，請重新提交。')
                record.pop('_path', None)
                record.pop('_last_frame', None)
            if record['status'] == 'expired' and now - record.get('finished_at', now) > 2 * REMOTE_RESULT_TTL:
                REMOTE_JOBS.pop(job_id, None)

'''
Sweep completed records periodically without interfering with inference.
定期整理已完成資料，不干擾單一 GPU worker 的推論。
'''
def remote_maintenance():
    while True:
        try:
            remote_cleanup()
        except Exception as error:
            print('API cleanup warning:', type(error).__name__, str(error)[:200])
        time.sleep(300)

'''
Serve only the generation API without exposing notebook controls.
只提供生成 API，不公開 Notebook 的管理功能。
'''
class RemoteHandler(BaseHTTPRequestHandler):
    def log_message(self, fmt, *args):
        return

    def send_json(self, code, payload, retry_after=None):
        data = json.dumps(payload, ensure_ascii=False).encode('utf-8')
        self.send_response(code)
        self.send_header('Content-Type', 'application/json; charset=utf-8')
        self.send_header('Content-Length', str(len(data)))
        if retry_after is not None:
            self.send_header('Retry-After', str(retry_after))
        self.end_headers()
        self.wfile.write(data)

    def do_GET(self):
        path = urlsplit(self.path).path
        if path == '/health':
            with REMOTE_JOB_LOCK:
                running = sum(record.get('status') == 'running' for record in REMOTE_JOBS.values())
            self.send_json(200, {'ok': True, 'api': 'kaggle-generation-bridge-v3',
                                 'backend_ready': remote_backend_ready(), 'stalled': remote_stalled(),
                                 'running': running, 'waiting': REMOTE_QUEUE.qsize(),
                                 'queue_capacity': REMOTE_QUEUE.maxsize})
            return
        if path == '/schema':
            self.send_json(200, {'kind': REMOTE_KIND, 'tasks': REMOTE_ALLOWED_TASKS,
                                 'fields': REMOTE_FIELDS, 'image_limit': REMOTE_MAX_IMAGES,
                                 'upload_field': 'images' if REMOTE_MAX_IMAGES else None,
                                 'queue_capacity': REMOTE_QUEUE.maxsize,
                                 'model_switching': 'configure in notebook first'})
            return
        pieces = path.strip('/').split('/')
        if pieces[0] != 'jobs' or len(pieces) not in (2, 3):
            self.send_json(404, {'error': 'Not found'})
            return
        job_id = pieces[1]
        with REMOTE_JOB_LOCK:
            record = REMOTE_JOBS.get(job_id)
            item = dict(record) if record else None
            if item and item['status'] == 'running':
                item['stalled'] = time.monotonic() - item.get('_start_clock', time.monotonic()) > REMOTE_STALL_SECONDS
            if item and item['status'] == 'queued':
                waiting = [key for key, value in REMOTE_JOBS.items() if value.get('status') == 'queued']
                if job_id in waiting:
                    item['queue_position'] = waiting.index(job_id) + 1
            if item and len(pieces) == 3 and pieces[2] in ('file', 'last-frame') and item['status'] == 'done':
                record['_downloads'] = record.get('_downloads', 0) + 1
                downloading = True
            else:
                downloading = False
        if item is None:
            self.send_json(404, {'error': 'Unknown job'})
            return
        if len(pieces) == 2:
            for field in tuple(item):
                if field.startswith('_'):
                    item.pop(field)
            self.send_json(200, item)
            return
        if not downloading:
            self.send_json(404, {'error': 'Result not available for this job'})
            return
        try:
            file_path = item.get('_path') if pieces[2] == 'file' else item.get('_last_frame')
            if not file_path or not Path(file_path).is_file():
                self.send_json(404, {'error': 'Output is no longer available'})
                return
            result = Path(file_path)
            self.send_response(200)
            self.send_header('Content-Type', 'application/octet-stream')
            self.send_header('Content-Disposition', f'attachment; filename="{result.name}"')
            self.send_header('Content-Length', str(result.stat().st_size))
            self.end_headers()
            with result.open('rb') as stream:
                shutil.copyfileobj(stream, self.wfile, length=1024 * 1024)
        except (BrokenPipeError, ConnectionResetError):
            pass
        finally:
            with REMOTE_JOB_LOCK:
                record['_downloads'] -= 1

    def do_POST(self):
        if urlsplit(self.path).path != '/generate':
            self.send_json(404, {'error': 'Not found'})
            return

        '''
        Reuse the same job ID for retry attempts after an uncertain response.
        網路回應不明時，重試會沿用同一個工作 ID，避免重複生成。
        '''
        supplied_id = self.headers.get('X-Request-ID', '').strip()
        try:
            job_id = uuid.UUID(supplied_id).hex if supplied_id else uuid.uuid4().hex
        except ValueError:
            self.send_json(400, {'error': 'X-Request-ID must be a UUID'})
            return
        with REMOTE_JOB_LOCK:
            existing = REMOTE_JOBS.get(job_id)
        if existing:
            self.send_json(202, {'job_id': job_id, 'status_url': f'/jobs/{job_id}',
                                 'download_url': f'/jobs/{job_id}/file', 'reused': True})
            return
        if remote_stalled():
            self.send_json(503, {'error': 'GPU worker appears stalled; inspect the Notebook before accepting more jobs'}, 60)
            return
        if not remote_backend_ready():
            self.send_json(503, {'error': 'Backend is not ready; execute model initialization cells first'}, 20)
            return
        if shutil.disk_usage(REMOTE_ROOT).free < REMOTE_MIN_FREE_BYTES:
            self.send_json(507, {'error': 'Kaggle disk space is low; clean old outputs before retrying'})
            return
        if not REMOTE_UPLOAD_SLOTS.acquire(timeout=1):
            self.send_json(503, {'error': 'Upload slots busy; retry later'}, 5)
            return
        staging = None
        try:
            self.connection.settimeout(120)
            length = int(self.headers.get('Content-Length', '0'))
            if not 0 < length <= 120 * 1024 * 1024:
                raise ValueError('Request must be between 1 byte and 120 MiB')
            body = self.rfile.read(length)
            if len(body) != length:
                raise ValueError('Incomplete request body')
            content_type = self.headers.get('Content-Type', '')
            fields, files = {}, []
            if content_type.startswith('application/json'):
                fields = json.loads(body)
            elif content_type.startswith('application/x-www-form-urlencoded'):
                fields = {key: values[-1] for key, values in parse_qs(body.decode('utf-8')).items()}
            elif content_type.startswith('multipart/form-data'):
                message = BytesParser(policy=policy.default).parsebytes(
                    b'Content-Type: ' + content_type.encode() + b'\r\nMIME-Version: 1.0\r\n\r\n' + body)
                for part in message.iter_parts():
                    name = part.get_param('name', header='content-disposition')
                    data = part.get_payload(decode=True)
                    if name == 'images':
                        files.append((part.get_filename() or 'upload.png', data))
                    elif name:
                        fields[name] = data.decode('utf-8')
            else:
                raise ValueError('Use application/json, form data, or multipart/form-data')
            if not isinstance(fields, dict):
                raise ValueError('Parameters must be a JSON object')
            if 'abc' in fields and len(str(fields['abc']).encode('utf-8')) > 1024 * 1024:
                raise ValueError('ABC score must be 1 MiB or smaller')
            task = str(fields.get('task', REMOTE_ALLOWED_TASKS[0])).lower()
            if task not in REMOTE_ALLOWED_TASKS:
                raise ValueError('Unsupported task: ' + task)
            if len(files) > REMOTE_MAX_IMAGES:
                raise ValueError(f'Only {REMOTE_MAX_IMAGES} input images are supported')
            if REMOTE_KIND in ('music', 'image_text') and files:
                raise ValueError('This model does not accept reference images')
            if REMOTE_KIND == 'video':
                minimum, maximum = {'t2v': (0, 0), 'i2v': (1, 1), 'flf2v': (2, 2),
                                    'r2v': (1, REMOTE_MAX_IMAGES)}[task] if task == 'r2v' else {
                                    't2v': (0, 0), 'i2v': (1, 1), 'flf2v': (2, 2)}[task]
                if not minimum <= len(files) <= maximum:
                    raise ValueError('Input image count does not match the selected video task')
            if REMOTE_KIND == 'image_multi':
                minimum, maximum = {'text': (0, 0), 'edit': (1, 1), 'multi': (2, REMOTE_MAX_IMAGES)}[task]
                if not minimum <= len(files) <= maximum:
                    raise ValueError(f'Task {task} requires {minimum}–{maximum} images')
            for key in fields:
                if key not in REMOTE_FIELDS:
                    raise ValueError('Unsupported parameter: ' + key)
            for filename, data in files:
                if not isinstance(data, bytes) or len(data) > 20 * 1024 * 1024:
                    raise ValueError('Each image must be 20 MiB or smaller')
                if Path(filename).suffix.lower() not in {'.png', '.jpg', '.jpeg', '.webp'}:
                    raise ValueError('Only PNG/JPEG/WebP images are supported')
                from PIL import Image
                with Image.open(io.BytesIO(data)) as photo:
                    photo.verify()
            staging = Path(tempfile.mkdtemp(prefix='pending_', dir=REMOTE_ROOT))
            image_names = []
            for index, (filename, data) in enumerate(files, 1):
                target = staging / f'{index:03d}{Path(filename).suffix.lower()}'
                target.write_bytes(data)
                image_names.append(target.name)

            '''
            Commit the validated upload and queue entry in one critical section.
            在同一個鎖定區段確認請求、保存資料並加入工作佇列。
            '''
            with REMOTE_JOB_LOCK:
                existing = REMOTE_JOBS.get(job_id)
                full = REMOTE_QUEUE.full()
                if not existing and not full:
                    folder = REMOTE_ROOT / job_id
                    staging.rename(folder)
                    staging = None
                    image_paths = [str(folder / name) for name in image_names]
                    REMOTE_JOBS[job_id] = {'id': job_id, 'task': task, 'status': 'queued',
                                           'created_at': time.time()}
                    REMOTE_QUEUE.put_nowait((job_id, fields, image_paths))
                    position = REMOTE_QUEUE.qsize()
            if existing:
                self.send_json(202, {'job_id': job_id, 'status_url': f'/jobs/{job_id}',
                                     'download_url': f'/jobs/{job_id}/file', 'reused': True})
            elif full:
                self.send_json(429, {'error': 'Queue is full; retry later'}, 15)
            else:
                self.send_json(202, {'job_id': job_id, 'status_url': f'/jobs/{job_id}',
                                     'download_url': f'/jobs/{job_id}/file', 'queue_position': position})
        except (ValueError, KeyError, TypeError, json.JSONDecodeError) as error:
            self.send_json(400, {'error': str(error)})
        except (TimeoutError, ConnectionError, OSError) as error:
            self.send_json(503, {'error': f'Upload temporarily failed: {type(error).__name__}'}, 10)
        except Exception as error:
            self.send_json(500, {'error': type(error).__name__ + ': ' + str(error)[:250]})
        finally:
            if staging is not None:
                shutil.rmtree(staging, ignore_errors=True)
            REMOTE_UPLOAD_SLOTS.release()

'''
Convert numeric HTTP form fields into explicitly supported generator settings.
只轉換白名單中的 HTTP 參數，不執行客戶端提供的任何程式碼。
'''
def remote_generate(job_id, fields, image_paths):
    with REMOTE_ENGINE_LOCK:
        with REMOTE_JOB_LOCK:
            REMOTE_JOBS[job_id].update(status='running', _start_clock=time.monotonic())
        namespace = globals()
        keys = ['TASK', 'PROMPT', 'WIDTH', 'HEIGHT', 'SEED', 'LM_SEED', 'SECONDS', 'STEPS', 'CFG',
                'LYRICS', 'TAGS', 'STYLE', 'DURATION', 'GPU_ID', 'COT', 'ABC_PATH',
                'OUTPUT_NAME', 'REFERENCE_PATHS', 'ACTIVE_JOB', 'TASK_CONFIGS',
                'R2V_REF_DIR', 'NEGATIVE_PROMPT', 'SAMPLER', 'REFERENCE_SIZE',
                'OUTPUTS', 'output', 'OUTPUT_PATH', 'OUTPUT_MP4', 'LAST_OUTPUT', 'RUN_SEED']
        before = {key: copy.deepcopy(namespace.get(key)) for key in keys if key in namespace}
        missing = set(keys) - set(before)
        try:
            task = fields.get('task', REMOTE_ALLOWED_TASKS[0]).lower()
            numerics = {'seed': int, 'lm_seed': int, 'width': int, 'height': int, 'steps': int,
                        'seconds': float, 'cfg': float, 'duration': int,
                        'image_strength': float, 'reference_size': int, 'audio_shift': float}
            inputs = {}
            for key, value in fields.items():
                inputs[key] = numerics[key](value) if key in numerics else value

            '''
            Resolve -1 once on the server so the exact value can be reproduced.
            在服務端一次決定 -1 的實際種子，方便重現結果。
            '''
            for seed_key in ('seed', 'lm_seed'):

                '''
                Ignore missing seeds and validate the supported 32-bit range.
                未提供的種子沿用 Notebook 設定，並檢查 32-bit 範圍。
                '''
                if seed_key in inputs:
                    if not -1 <= inputs[seed_key] < 2**32:
                        raise ValueError(f'{seed_key} must be -1 or between 0 and 4294967295')

                    '''
                    Translate -1 into an explicit random value before inference.
                    推論前將 -1 轉成明確的隨機數值。
                    '''
                    if inputs[seed_key] == -1:
                        inputs[seed_key] = secrets.randbelow(2**32)

            '''
            Save supplied seeds for reproducibility in the job status API.
            在工作狀態 API 中記錄請求所使用的實際種子。
            '''
            with REMOTE_JOB_LOCK:
                for seed_key in ('seed', 'lm_seed'):
                    if seed_key in inputs:
                        REMOTE_JOBS[job_id][seed_key + '_used'] = inputs[seed_key]
            for dimension in ('width', 'height'):
                if dimension in inputs and not 0 < inputs[dimension] <= 4096:
                    raise ValueError('Width and height must be between 1 and 4096')
            for key in ('seconds', 'duration'):
                if key in inputs and not 0 < inputs[key] <= 3600:
                    raise ValueError('Duration must be between 0 and 3600 seconds')
            if 'reference_size' in inputs and not 64 <= inputs['reference_size'] <= 2048:
                raise ValueError('Reference size must be between 64 and 2048')
            if 'ref_image_size' in inputs and inputs['ref_image_size'] not in ('match', 'max'):
                raise ValueError('Reference image mode must be match or max')
            if 'image_strength' in inputs and not 0 <= inputs['image_strength'] <= 1:
                raise ValueError('Image strength must be between 0 and 1')
            if 'steps' in inputs and not 1 <= inputs['steps'] <= 200:
                raise ValueError('Steps must be between 1 and 200')
            if REMOTE_KIND == 'video':
                if 'TASK_CONFIGS' in namespace:
                    if task not in namespace['TASK_CONFIGS']:
                        raise ValueError('Run this task configuration cell before API generation')
                    updated = copy.deepcopy(namespace['TASK_CONFIGS'][task])
                    for field in ('prompt', 'width', 'height', 'seconds', 'steps', 'seed',
                                  'audio_shift', 'ref_image_size'):
                        if field in inputs:
                            updated[field] = inputs[field]
                    updated['output_name'] = f'remote_{job_id[:12]}.mp4'
                    if task == 'i2v':
                        updated['first_image'] = image_paths[0]
                    elif task == 'flf2v':
                        updated['first_image'], updated['last_image'] = image_paths
                    elif task == 'r2v':
                        updated['ref_images'] = image_paths
                        '''
                        Point R2V to this job's uploaded reference images only.
                        讓 R2V 僅讀取這筆工作的參考圖片資料夾。
                        '''
                        namespace['R2V_REF_DIR'] = Path(image_paths[0]).parent
                    namespace['TASK_CONFIGS'][task] = updated
                    namespace['TASK'] = task
                else:

                    '''
                    LTX-2.5 uses a shared ACTIVE_JOB configuration.
                    LTX-2.5 使用共用 ACTIVE_JOB 參數。
                    '''
                    namespace['ACTIVE_JOB'] = {'mode': task, 'prompt': inputs.get('prompt', namespace.get('PROMPT', '')),
                                               'run_name': f'remote_{job_id[:12]}',
                                               'first_path': image_paths[0] if image_paths else None,
                                               'last_path': image_paths[1] if len(image_paths) > 1 else None}
                    for key in ('width', 'height', 'seconds', 'seed', 'image_strength'):
                        if key in inputs:
                            namespace[key.upper()] = inputs[key]
            else:
                for key in ('prompt', 'width', 'height', 'seed', 'lm_seed', 'steps', 'cfg',
                            'lyrics', 'tags', 'style', 'duration', 'negative_prompt',
                            'sampler', 'reference_size', 'cot'):
                    if key in inputs:
                        namespace[key.upper()] = inputs[key]
                if REMOTE_KIND == 'image_multi':
                    namespace['TASK'] = task
                    namespace['REFERENCE_PATHS'] = image_paths
                if REMOTE_KIND == 'music':
                    namespace['OUTPUT_NAME'] = f'remote_{job_id[:12]}.mp3'
                    if 'ABC_PATH' in namespace:
                        namespace['ABC_PATH'] = ''
                    if 'abc' in inputs:
                        score = (REMOTE_ROOT / job_id / 'input.abc')
                        score.write_text(str(inputs['abc']), encoding='utf-8')
                        namespace['ABC_PATH'] = str(score)
                if REMOTE_KIND in ('image_multi', 'image_text'):
                    namespace['OUTPUT_NAME'] = f'remote_{job_id[:12]}.png'

            '''
            Clear old output metadata so a failed generation cannot reuse another job's file.
            清除前次輸出狀態，避免本次失敗時誤取其他工作的結果。
            '''
            for output_key in ('OUTPUTS', 'output', 'OUTPUT_PATH', 'OUTPUT_MP4', 'LAST_OUTPUT'):
                namespace.pop(output_key, None)

            '''
            Execute a fixed local generation cell, never user-supplied code.
            僅執行 Notebook 內固定的生成程式，不會執行請求傳來的原始碼。
            '''
            exec(compile(REMOTE_SCRIPT, '<notebook-generation-cell>', 'exec'), namespace)
            paths = []
            if REMOTE_KIND == 'video':
                paths = [namespace.get('LAST_OUTPUT') or namespace.get('OUTPUT_MP4')]
            elif REMOTE_KIND == 'image_text':
                paths = [namespace.get('OUTPUT_PATH')]
            elif REMOTE_KIND == 'image_multi':
                paths = [namespace.get('output')]
            elif REMOTE_KIND == 'music':
                paths = namespace.get('OUTPUTS', []) if 'OUTPUTS' in namespace else [namespace.get('output')]
            results = [Path(path) for path in paths if path is not None]
            if REMOTE_KIND == 'music' and REMOTE_SERVICE_ID.startswith('06_YuE2_'):

                '''
                Include YuE2 replay metadata and score with the single MP3.
                將 YuE2 的重播參數與 ABC 樂譜一起提供下載。
                '''
                results += [sidecar for path in list(results)
                            for ext in ('.json', '.abc')
                            if (sidecar := path.with_suffix(ext)).is_file()]
            if not results or any(not path.is_file() for path in results):
                raise FileNotFoundError('Generation finished without a saved result file')
            folder = REMOTE_ROOT / job_id
            if len(results) == 1:
                target = folder / f'result{results[0].suffix.lower()}'
                shutil.copy2(results[0], target)
            else:
                target = folder / 'results.zip'
                with zipfile.ZipFile(target, 'w', compression=zipfile.ZIP_STORED) as archive:
                    for result in results:
                        archive.write(result, arcname=result.name)
            last_frame = None
            if REMOTE_KIND == 'video' and shutil.which('ffprobe') and shutil.which('ffmpeg'):
                try:
                    probe = subprocess.run(['ffprobe', '-v', 'error', '-count_frames',
                                            '-select_streams', 'v:0', '-show_entries',
                                            'stream=nb_read_frames', '-of', 'json', str(target)],
                                           check=True, text=True, capture_output=True, timeout=120)
                    index = int(json.loads(probe.stdout)['streams'][0]['nb_read_frames']) - 1
                    if index >= 0:
                        last_frame = folder / 'last_frame.png'
                        subprocess.run(['ffmpeg', '-y', '-loglevel', 'error', '-i', str(target),
                                        '-vf', rf'select=eq(n\,{index})', '-frames:v', '1',
                                        str(last_frame)], check=True)
                except (OSError, ValueError, KeyError, subprocess.CalledProcessError, subprocess.TimeoutExpired):
                    last_frame = None

            '''
            Remove only source files bearing this remote job's unique prefix.
            僅清理本次遠端工作專屬原始輸出，保留 Notebook 手動生成的檔案。
            '''
            for original in results:
                if original.name.startswith(f'remote_{job_id[:12]}') and original != target:
                    try:
                        original.unlink(missing_ok=True)
                    except OSError:
                        pass
            comfy = globals().get('COMFY')
            if comfy is not None and (Path(comfy) / 'output').exists():
                for artifact in (Path(comfy) / 'output').rglob(f'remote_{job_id[:12]}*'):
                    if artifact.is_file():
                        try:
                            artifact.unlink()
                        except OSError:
                            pass
            with REMOTE_JOB_LOCK:
                REMOTE_JOBS[job_id].update(status='done', filename=target.name,
                                           bytes=target.stat().st_size, _path=str(target),
                                           last_frame_available=bool(last_frame),
                                           _last_frame=str(last_frame) if last_frame else None)
        except Exception as error:
            with REMOTE_JOB_LOCK:
                REMOTE_JOBS[job_id].update(status='error', error=f'{type(error).__name__}: {error}'[:2000])
        finally:
            for key in missing:
                namespace.pop(key, None)
            namespace.update(before)

'''
Use one GPU worker, regardless of the number of HTTP request threads.
不論有多少 HTTP 連線，都只有一個 GPU worker 消費佇列。
'''
def remote_worker():
    while True:
        job_id, fields, image_paths = REMOTE_QUEUE.get()
        try:
            remote_generate(job_id, fields, image_paths)
        except Exception as error:
            with REMOTE_JOB_LOCK:
                if job_id in REMOTE_JOBS:
                    REMOTE_JOBS[job_id].update(status='error', error=f'{type(error).__name__}: {error}'[:2000])
        finally:
            with REMOTE_JOB_LOCK:
                if job_id in REMOTE_JOBS:
                    REMOTE_JOBS[job_id]['finished_at'] = time.time()
            try:
                remote_remove_inputs(job_id)
            except OSError as error:
                print('API input cleanup warning:', str(error)[:180])
            finally:
                REMOTE_QUEUE.task_done()

'''
Keep one worker and one maintenance thread even after rerunning this cell.
重跑 Cell 仍僅保留一個 worker 和一個清理 thread。
'''
if not (globals().get('REMOTE_WORKER') and REMOTE_WORKER.is_alive()):
    REMOTE_WORKER = threading.Thread(target=remote_worker, daemon=True)
    REMOTE_WORKER.start()
if not (globals().get('REMOTE_MAINTENANCE') and REMOTE_MAINTENANCE.is_alive()):
    REMOTE_MAINTENANCE = threading.Thread(target=remote_maintenance, daemon=True)
    REMOTE_MAINTENANCE.start()

'''
Close only the previous ngrok tunnel; preserve the running API and its jobs.
只關閉舊的 ngrok 通道，保留正在運作的 API 與所有生成工作。
'''
old_tunnel = globals().get('REMOTE_NGROK_PROC')
if old_tunnel is not None and old_tunnel.poll() is None:
    old_tunnel.terminate()
    try:
        old_tunnel.wait(timeout=5)
    except subprocess.TimeoutExpired:
        old_tunnel.kill()
        old_tunnel.wait(timeout=5)
NGROK_URL = ''

'''
Install the official ngrok agent only if no usable binary exists.
系統尚無 ngrok 才下載官方執行檔，不重複安裝。
'''
ngrok_bin = shutil.which('ngrok') or str(Path('/kaggle/working/ngrok'))
if not Path(ngrok_bin).is_file():
    url = 'https://bin.ngrok.com/c/bNyj1mQVY4c/ngrok-v3-stable-linux-amd64.tgz'
    with urllib.request.urlopen(url, timeout=90) as response:
        blob = response.read()
    with tarfile.open(fileobj=io.BytesIO(blob), mode='r:gz') as archive:
        entry = next((member for member in archive.getmembers() if Path(member.name).name == 'ngrok' and member.isfile()), None)
        if entry is None:
            raise RuntimeError('ngrok 官方下載檔案缺少執行檔')
        Path(ngrok_bin).write_bytes(archive.extractfile(entry).read())
    Path(ngrok_bin).chmod(0o755)

'''
Always request the auth token again when this cell is rerun.
每次重跑均重新要求輸入 ngrok Authtoken，不顯示內容。
'''
ngrok_token = getpass('輸入 ngrok Authtoken：').strip()
if not ngrok_token:
    raise ValueError('未輸入 ngrok Authtoken')
result = subprocess.run([ngrok_bin, 'config', 'add-authtoken', ngrok_token], capture_output=True, text=True)
if result.returncode:
    raise RuntimeError('ngrok token 設定失敗（請確認 Token 是否有效）')
del ngrok_token
result = subprocess.run([ngrok_bin, 'config', 'check'], capture_output=True, text=True)
if result.returncode:
    raise RuntimeError('ngrok 設定格式檢查失敗：' + result.stderr[-500:])

'''
Reuse the same API server when this cell is executed again.
重跑本格時沿用同一個 API Server，不中斷生成工作或清空佇列。
'''
server = globals().get('REMOTE_SERVER')
server_thread = globals().get('REMOTE_SERVER_THREAD')
server_is_current = (
    server is not None
    and server_thread is not None
    and server_thread.is_alive()
    and getattr(server, 'notebook_id', None) == REMOTE_SERVICE_ID
)
if server is not None and not server_is_current:
    if server_thread is not None and server_thread.is_alive():
        server.shutdown()
    server.server_close()
if not server_is_current:
    try:
        REMOTE_SERVER = ThreadingHTTPServer(('127.0.0.1', REMOTE_PORT), RemoteHandler)
    except OSError as error:
        raise RuntimeError(f'本機 API Port {REMOTE_PORT} 已被其他服務占用') from error
    REMOTE_SERVER.notebook_id = REMOTE_SERVICE_ID
    REMOTE_SERVER.daemon_threads = True
    REMOTE_SERVER_THREAD = threading.Thread(target=REMOTE_SERVER.serve_forever, daemon=True)
    REMOTE_SERVER_THREAD.start()
    print('已啟動新的本機 API Server。')
else:

    '''
    Update the handler on rerun without restarting an active GPU job.
    重跑後更新 API handler，但不重新啟動正在執行的生成工作。
    '''
    REMOTE_SERVER.RequestHandlerClass = RemoteHandler
    print('已沿用本機 API Server；現有生成工作與結果保持不變。')
with urllib.request.urlopen(f'http://127.0.0.1:{REMOTE_PORT}/health', timeout=4) as response:
    if json.load(response).get('ok') is not True:
        raise RuntimeError('本機 API Server 尚未就緒')

'''
Use the agent inspection API to obtain the actual public URL.
透過 ngrok 本機檢查 API 取得實際對外 HTTPS 網址。
'''
NGROK_LOG = REMOTE_ROOT / 'ngrok.log'
with NGROK_LOG.open('w') as log:
    REMOTE_NGROK_PROC = subprocess.Popen([ngrok_bin, 'http', str(REMOTE_PORT)], stdout=log,
                                         stderr=subprocess.STDOUT, start_new_session=True)
NGROK_URL = ''
for attempt in range(25):
    if REMOTE_NGROK_PROC.poll() is not None:
        break
    try:
        with urllib.request.urlopen('http://127.0.0.1:4040/api/tunnels', timeout=2) as response:
            tunnels = json.load(response).get('tunnels', [])
        NGROK_URL = next((item['public_url'] for item in tunnels
                          if item.get('public_url', '').startswith('https://') and
                          str(REMOTE_PORT) in item.get('config', {}).get('addr', '')), '')
        if NGROK_URL:
            break
    except (OSError, ValueError, KeyError):
        pass
    time.sleep(1)
if not NGROK_URL:
    raise RuntimeError('ngrok URL 建立失敗：\n' + NGROK_LOG.read_text(errors='replace')[-2500:])
print('ngrok HTTPS URL:', NGROK_URL)
print('提醒：公開的 ngrok API URL 不需要額外驗證，請只分享給授課學生。')
print('GET /health | GET /schema | POST /generate | GET /jobs/{id} | GET /jobs/{id}/file')
print('影片輸出另提供：GET /jobs/{id}/last-frame')


## 13. Web API 使用說明

In [ ]:
'''
Query the active bridge for supported parameters and task names.
從正在執行的橋接服務讀取實際支援的參數與模式。
'''
import json
import urllib.request

'''
Request the API schema directly from the local API server.
直接從本機 API Server 查詢支援的參數與模式。
'''
request = urllib.request.Request(f'http://127.0.0.1:{REMOTE_PORT}/schema')
with urllib.request.urlopen(request, timeout=5) as response:
    schema = json.load(response)
print(json.dumps(schema, ensure_ascii=False, indent=2))
print('用法：POST NGROK_URL + "/generate"（JSON 或 multipart/images），取得 job_id；')
print('輪詢 GET /jobs/{job_id}；完成後下載 GET /jobs/{job_id}/file。')
print('本地 requests 範例：06_LTX-2.5_影片生成_Kaggle_client.py（與本 Notebook 同名對應）。')
print('使用前只需在對應的 client.py 填入 API_URL。')
print('本服務使用單一 FIFO worker，遠端請求不會同時啟動多筆 GPU 推論。')